# Loading the Dataset

## Initial Sample, Sample Size: 350

In [23]:
from datasets import load_dataset

ds = load_dataset("zefang-liu/phishing-email-dataset", split="train")
ds = ds.class_encode_column("Email Type")

split1 = ds.train_test_split(test_size=350, stratify_by_column="Email Type", seed=20)
sample1 = split1["test"]

sample1[0]

{'Unnamed: 0': 1869,
 'Email Text': 'term paper stuart , i cannot open your attachment . vince kaminski',
 'Email Type': 1}

Email Type: 0 == Phishing

Email Type: 1 == Safe

## Fetching emails to use in few-shot prompting

In [24]:
split2 = split1['train'].train_test_split(test_size=4, stratify_by_column='Email Type', seed=20)

leftover_emails = split2['train']

random_emails_4 = split2['test']

random_emails_4[1]

{'Unnamed: 0': 3905,
 'Email Text': "Ok, Iknow this is blatantly OT but I'm beginning to go insane.\nHad an old Dell Dimension XPS sitting in the corner and decided to\nput it to use, I know it was working pre being stuck in the\ncorner, but when I plugged it in, hit the power nothing happened.\nI opened her up and had a look and say nothing much. A little orange\nLED comes on when I plug her in but that's it, after some googling\nI found some reference to re-seating all the parts, but no change.\nThe problem I'm having is that since the power supply is some Dell\nspecific one, ATX block with what looks like one of the old AT\npower connectors, I cant figure out weather this is a Mobo prob\nor a PSU prob. Just to futily try and drag this back OT, I want\nto install Linux on it when I get it working. If anyone knows\nwhat the problem might be give me a shout.Cheers,\nPeter.--\nPeter Aherne, Software Engineer, \nMotorola Ireland Ltd.\nPh: +353 21 4511234 Mobile: +353 87 2246834-- \nIrish

## Final Sample, Sample Size: 500

In [ ]:
split3 = leftover_emails.train_test_split(test_size=2, stratify_by_column='Email Type', seed=20)

# leftover_emails = split3['train']

sample2 = split3['test']

sample2[1]

{'Unnamed: 0': 16606,
 'Email Text': 're : 6 . 293 words that are their own opposites sue morrish \'s posting on shame in australian reminded me of a weird fact about the same word in south african english , according to some sa friends . this is that " shame ! " is used as an exclamation of joy by , for example , old ladies seeing a newborn baby or a fluffy animal . the supposed explanation is that " shame ! " as an exclamation of disapproval became an exclamation of sympathy for somebody who has been ill-treated ( so far , this parallels a shift that i \' m familiar with too ) . it then bleached out still further in sa to a mere " back channel utterance " , indicating that the listener was still paying ( sympathetic ) attention , and then became a positive expression of pleasure . can anybody confirm either the data or the explanation ? btw , wrt benji wald \'s posting , i understood the origin of a \' lucus a non lucendo \' to be st isidore of seville \'s _ origines sive etymologiae

# Evaluation Metrics Calculator

In [26]:
import csv
import os

def evaluate_models(model_list, save_folder):
    print(f"===Evaluating models: {model_list}===")
    os.makedirs(save_folder, exist_ok=True)

    model_comparison_file = os.path.join(save_folder, "model_comparison.csv")
    with open(model_comparison_file, mode='w', newline='') as comparison_file:
        writer = csv.writer(comparison_file)
        writer.writerow(["Model", "Accuracy", "Precision", "Recall", "F1 Score"])

        for model_name in model_list:

            fields = []
            prediction_categories = {
                'TP': 0,
                'FP': 0,
                'TN': 0,
                'FN': 0
            }
            metrics = {
                'Accuracy': None,
                'Precision': None,
                'Recall': None,
                'F1 Score': None
            }
                
            csv_file = os.path.join(save_folder, f"{model_name.replace(':', '_')}.csv")
            with open(csv_file, mode='r') as file:
                reader = csv.reader(file)

                fields = next(reader)
                total_rows = 0
                for row in reader:
                    if row[2] != "ERROR":
                        total_rows += 1

                    if row[1] == '0' and row[2] == 'True':
                        prediction_categories['TP'] += 1
                    elif row[1] == '0' and row[2] == 'False':
                        prediction_categories['FN'] += 1
                    elif row[1] == '1' and row[2] == 'True':
                        prediction_categories['FP'] += 1
                    elif row[1] == '1' and row[2] == 'False':
                        prediction_categories['TN'] += 1
            
            metrics['Accuracy'] = (prediction_categories['TP'] + prediction_categories['TN']) / total_rows
            metrics['Precision'] = prediction_categories['TP'] / (prediction_categories['TP'] + prediction_categories['FP'])
            metrics['Recall'] = prediction_categories['TP'] / (prediction_categories['TP'] + prediction_categories['FN'])
            metrics['F1 Score'] = 2 * (metrics['Precision'] * metrics['Recall']) / (metrics['Precision'] + metrics['Recall'])

            print(f"\nModel: {model_name}. Results evaluated: {total_rows}.\n")
            for key in metrics:
                print(f"\t{key}: {metrics[key]}")

            writer.writerow([model_name, metrics['Accuracy'], metrics['Precision'], metrics['Recall'], metrics['F1 Score']])

    print(f"\n===Model evaluation complete. Comparison saved to '{model_comparison_file}'===")


# Iteration 1: Initial Evaluation

In [ ]:
import requests
import csv
import json
import os
from tqdm import tqdm


def test_models(email_list, model_list, save_folder):
    api_url = "http://localhost:11434/api/generate"
    os.makedirs(save_folder, exist_ok=True)

    print(f"===Models selected: {model_list}===")

    for model_name in model_list:
        print(f'\nStarting phishing email analysis...\nEmail count: {email_list.num_rows}\nModel: {model_name}\n')
        
        csv_file = os.path.join(save_folder, f"{model_name.replace(':', '_')}.csv")
        with open(csv_file, mode='w', newline='') as file:
            writer = csv.writer(file)
            writer.writerow(["Email", "Label", "is_phishing_verdict", "risk_level", "explanation"])
            
            for i in tqdm(range(email_list.num_rows)):
                email = email_list[i]

                max_retries = 5
                success = False

                for attempt in range(max_retries):
                    payload = {
                        "model": model_name,
                        "prompt": f"""
                                                    Analyse this email for phishing indicators.
                            Email: "{email}"
                            
                            Return a raw JSON object with this structure:
                            {{
                                "is_phishing": boolean,
                                "risk_level": "Low" | "Medium" | "High",
                                "explanation": "Concise reason why."
                            }}
                        """,
                        "format": "json",  # Forces local model into JSON mode
                        "stream": False,
                        "options": {
                            "temperature": 0.0,
                            "seed": 24,
                            "num_ctx": 4096
                        }
                    }

                    try:
                        response = requests.post(api_url, json=payload)
                        response.raise_for_status()
                        
                        raw_response = response.json().get("response", "").strip()
                        thinking_text = response.json().get("thinking", "").strip()

                        if raw_response != "":
                            json_string= raw_response

                        else:
                            json_string = thinking_text

                        verdict = json.loads(json_string)

                        writer.writerow([email['Unnamed: 0'], email['Email Type'], verdict['is_phishing'], verdict.get('risk_level', 'Missing'), verdict.get('explanation', 'Missing')])

                        success = True
                        break  # Exit retry loop on success

                    except requests.exceptions.RequestException as e:
                        print(f"API Error on email {i} (Attempt {attempt+1}/{max_retries}): {e}")
                        
                    except json.JSONDecodeError as e:
                        print(f"JSON Error on email {i} (Attempt {attempt+1}/{max_retries}). Hallucination: {json_string[:50]}...")
                        
                    except Exception as e:
                        print(f"Unexpected Error on email {i} (Attempt {attempt+1}/{max_retries}): {e}")
                
                # If we failed 3 times, write a blank/error row so we don't lose alignment in our CSV
                if not success:
                    print(f"-> Skipping email {i} after {max_retries} failed attempts.")
                    writer.writerow([email['Unnamed: 0'], email['Email Type'], "ERROR", "ERROR", "Failed after max retries"])
    
        print(f"\nAnalysis complete.\nResults saved to '{csv_file}'\n")



In [ ]:
model_list = ['gemma3:4b', 'llama3:latest', 'qwen3.5:4b', 'gemma3:12b']
email_list = sample2
save_folder = "Model Evaluation 1 - Initial Stage"

test_models(email_list, model_list, save_folder)
evaluate_models(model_list, save_folder)

# Iteration 2: Two-Shot Prompting

In [ ]:
import requests
import csv
import json
import os
from tqdm import tqdm


def test_models(email_list, model_list, save_folder):
    api_url = "http://localhost:11434/api/generate"
    os.makedirs(save_folder, exist_ok=True)

    print(f"===Models selected: {model_list}===")

    for model_name in model_list:
        print(f'\nStarting phishing email analysis...\nEmail count: {email_list.num_rows}\nModel: {model_name}\n')
        
        csv_file = os.path.join(save_folder, f"{model_name.replace(':', '_')}.csv")
        with open(csv_file, mode='w', newline='') as file:
            writer = csv.writer(file)
            writer.writerow(["Email", "Label", "is_phishing_verdict", "risk_level", "explanation"])
            
            for i in tqdm(range(email_list.num_rows)):
                email = email_list[i]

                max_retries = 5
                success = False

                for attempt in range(max_retries):
                    payload = {
                        "model": model_name,
                        "prompt": f"""
                                You are an expert cybersecurity analyst. Your task is to analyze an email and determine if it is phishing or safe.
                                You must return a raw JSON object and absolutely nothing else.

                                Here are examples of how to classify emails:

                                --- EXAMPLE 1: Safe Email ---
                                Email: "Ok, Iknow this is blatantly OT but I'm beginning to go insane.\nHad an old Dell Dimension XPS sitting in the corner and decided to\nput it to use, I know it was working pre being stuck in the\ncorner, but when I plugged it in, hit the power nothing happened.\nI opened her up and had a look and say nothing much. A little orange\nLED comes on when I plug her in but that's it, after some googling\nI found some reference to re-seating all the parts, but no change.\nThe problem I'm having is that since the power supply is some Dell\nspecific one, ATX block with what looks like one of the old AT\npower connectors, I cant figure out weather this is a Mobo prob\nor a PSU prob. Just to futily try and drag this back OT, I want\nto install Linux on it when I get it working. If anyone knows\nwhat the problem might be give me a shout.Cheers,\nPeter.--\nPeter Aherne, Software Engineer, \nMotorola Ireland Ltd.\nPh: +353 21 4511234 Mobile: +353 87 2246834-- \nIrish Linux Users' Group: ilug@linux.ie\nhttp://www.linux.ie/mailman/listinfo/ilug for (un)subscription information.\nList maintainer: listmaster@linux.ie"
                                Output:
                                {{
                                    "is_phishing": false,
                                    "risk_level": "Low",
                                    "explanation": "Legitimate technical support inquiry sent to a public mailing list. It contains a verifiable signature block with standard contact details, lacks any fabricated urgency or requests for sensitive information, and the included URL is a standard, safe mailing list administrative link."
                                }}

                                --- EXAMPLE 2: Phishing Email ---
                                Email: 'emailing : mon , 30 aug 2004 19 : 28 : 52 - 0800 dear sir / madam ; from our records we understand that you are inquiring about a new profession . we have a limited , ont time offer . our university can offer you a pre - qualified degree in your field of choice . we offer signing bonuses of up to $ 15 , 000 in your profession . to obtain your degree with valid transcripts & information on new career bonusus , follow our link : sincerely , dr . julie paige administration office this communication is of private matter . if you have received this and ar enot the individual or group it may concern or show interest in this please follow so we know http : / / 1 highereducation . com / st . html and the proper measures will proficiently expidited in a timely manner .'
                                Output:
                                {{
                                    "is_phishing": true,
                                    "risk_level": "High",
                                    "explanation": "Classic scam lure. Uses a generic greeting, contains multiple spelling and grammatical errors ('ont time', 'bonusus', 'expidited'), offers highly unrealistic financial incentives for a 'pre-qualified degree', and directs the victim to a suspicious, unofficial domain."
                                }}

                                --- ACTUAL TASK ---
                                Analyse this email for phishing indicators.
                                Email: "{email['Email Text']}"

                                Return a raw JSON object with this exact structure:
                                {{
                                    "is_phishing": boolean,
                                    "risk_level": "Low" | "Medium" | "High",
                                    "explanation": "Concise reason why."
                                }}
                        """,
                        "format": "json",  # Forces local model into JSON mode
                        "stream": False,
                        "options": {
                            "temperature": 0.0,
                            "seed": 24,
                            "num_ctx": 4096
                        }
                    }

                    try:
                        response = requests.post(api_url, json=payload)
                        response.raise_for_status()
                        
                        raw_response = response.json().get("response", "").strip()
                        thinking_text = response.json().get("thinking", "").strip()

                        if raw_response != "":
                            json_string= raw_response

                        else:
                            json_string = thinking_text

                        verdict = json.loads(json_string)

                        writer.writerow([email['Unnamed: 0'], email['Email Type'], verdict['is_phishing'], verdict.get('risk_level', 'Missing'), verdict.get('explanation', 'Missing')])

                        success = True
                        break  # Exit retry loop on success

                    except requests.exceptions.RequestException as e:
                        print(f"API Error on email {i} (Attempt {attempt+1}/{max_retries}): {e}")
                        
                    except json.JSONDecodeError as e:
                        print(f"JSON Error on email {i} (Attempt {attempt+1}/{max_retries}). Hallucination: {json_string[:50]}...")
                        
                    except Exception as e:
                        print(f"Unexpected Error on email {i} (Attempt {attempt+1}/{max_retries}): {e}")
                
                # If we failed 3 times, write a blank/error row so we don't lose alignment in our CSV
                if not success:
                    print(f"-> Skipping email {i} after {max_retries} failed attempts.")
                    writer.writerow([email['Unnamed: 0'], email['Email Type'], "ERROR", "ERROR", "Failed after max retries"])
    
        print(f"\nAnalysis complete.\nResults saved to '{csv_file}'\n")



In [ ]:
model_list = ['gemma3:4b', 'llama3:latest', 'qwen3.5:4b', 'gemma3:12b']
email_list = sample2
save_folder = "Model Evaluation 2 - Few-Shot Stage"

test_models(email_list, model_list, save_folder)
evaluate_models(model_list, save_folder)

# Iteration 3: System Prompt Parameter, Increasing Precision and JSON Order

In [ ]:
import requests
import csv
import json
import os
from tqdm import tqdm


def test_models(email_list, model_list, save_folder):
    api_url = "http://localhost:11434/api/generate"
    os.makedirs(save_folder, exist_ok=True)

    print(f"===Models selected: {model_list}===")

    for model_name in model_list:
        print(f'\nStarting phishing email analysis...\nEmail count: {email_list.num_rows}\nModel: {model_name}\n')
        
        csv_file = os.path.join(save_folder, f"{model_name.replace(':', '_')}.csv")
        with open(csv_file, mode='w', newline='') as file:
            writer = csv.writer(file)
            writer.writerow(["Email", "Label", "is_phishing_verdict", "risk_level", "explanation"])
            
            for i in tqdm(range(email_list.num_rows)):
                email = email_list[i]

                max_retries = 5
                success = False

                for attempt in range(max_retries):
                    payload = {
                        "model": model_name,
                        "system": """
                            You are an expert cybersecurity analyst. Your only task is to analyze an email and determine if it is phishing or safe.
                            
                            CRITICAL RULES:
                            1. Do not flag automated newsletters or poor grammar as phishing unless there is clear malicious intent.
                            2. Look for credential harvesting, urgent financial requests, or suspicious URLs.
                            3. If ambiguous, default to Safe ("is_phishing": false).
                            4. You must return a raw JSON object and absolutely nothing else.
                        """,
                        "prompt": f"""
                            You are an expert cybersecurity analyst. Your task is to analyze an email and determine if it is phishing or safe.
                            You must return a raw JSON object and absolutely nothing else.

                            Here are examples of how to classify emails:

                            --- EXAMPLE 1: Safe Email ---
                            Email: "Ok, Iknow this is blatantly OT but I'm beginning to go insane.\nHad an old Dell Dimension XPS sitting in the corner and decided to\nput it to use, I know it was working pre being stuck in the\ncorner, but when I plugged it in, hit the power nothing happened.\nI opened her up and had a look and say nothing much. A little orange\nLED comes on when I plug her in but that's it, after some googling\nI found some reference to re-seating all the parts, but no change.\nThe problem I'm having is that since the power supply is some Dell\nspecific one, ATX block with what looks like one of the old AT\npower connectors, I cant figure out weather this is a Mobo prob\nor a PSU prob. Just to futily try and drag this back OT, I want\nto install Linux on it when I get it working. If anyone knows\nwhat the problem might be give me a shout.Cheers,\nPeter.--\nPeter Aherne, Software Engineer, \nMotorola Ireland Ltd.\nPh: +353 21 4511234 Mobile: +353 87 2246834-- \nIrish Linux Users' Group: ilug@linux.ie\nhttp://www.linux.ie/mailman/listinfo/ilug for (un)subscription information.\nList maintainer: listmaster@linux.ie"
                            Output:
                            {{
                                "is_phishing": false,
                                "risk_level": "Low",
                                "explanation": "Legitimate technical support inquiry sent to a public mailing list. It contains a verifiable signature block with standard contact details, lacks any fabricated urgency or requests for sensitive information, and the included URL is a standard, safe mailing list administrative link."
                            }}

                            --- EXAMPLE 2: Phishing Email ---
                            Email: 'emailing : mon , 30 aug 2004 19 : 28 : 52 - 0800 dear sir / madam ; from our records we understand that you are inquiring about a new profession . we have a limited , ont time offer . our university can offer you a pre - qualified degree in your field of choice . we offer signing bonuses of up to $ 15 , 000 in your profession . to obtain your degree with valid transcripts & information on new career bonusus , follow our link : sincerely , dr . julie paige administration office this communication is of private matter . if you have received this and ar enot the individual or group it may concern or show interest in this please follow so we know http : / / 1 highereducation . com / st . html and the proper measures will proficiently expidited in a timely manner .'
                            Output:
                            {{
                                "is_phishing": true,
                                "risk_level": "High",
                                "explanation": "Classic scam lure. Uses a generic greeting, contains multiple spelling and grammatical errors ('ont time', 'bonusus', 'expidited'), offers highly unrealistic financial incentives for a 'pre-qualified degree', and directs the victim to a suspicious, unofficial domain."
                            }}

                            --- ACTUAL TASK ---
                            Analyse this email for phishing indicators.
                            Email: "{email['Email Text']}"

                            Return a raw JSON object with this exact structure:
                            {{
                                "explanation": "Step-by-step reasoning of the technical and social indicators...",
                                "risk_level": "Low" | "Medium" | "High",
                                "is_phishing": boolean
                            }}
                        """,
                        "format": "json",  # Forces local model into JSON mode
                        "stream": False,
                        "options": {
                            "temperature": 0.0,
                            "seed": 24,
                            "num_ctx": 4096
                        }
                    }

                    try:
                        response = requests.post(api_url, json=payload)
                        response.raise_for_status()
                        
                        raw_response = response.json().get("response", "").strip()
                        thinking_text = response.json().get("thinking", "").strip()

                        if raw_response != "":
                            json_string= raw_response

                        else:
                            json_string = thinking_text

                        verdict = json.loads(json_string)

                        writer.writerow([email['Unnamed: 0'], email['Email Type'], verdict['is_phishing'], verdict.get('risk_level', 'Missing'), verdict.get('explanation', 'Missing')])

                        success = True
                        break  # Exit retry loop on success

                    except requests.exceptions.RequestException as e:
                        print(f"API Error on email {i} (Attempt {attempt+1}/{max_retries}): {e}")
                        
                    except json.JSONDecodeError as e:
                        print(f"JSON Error on email {i} (Attempt {attempt+1}/{max_retries}). Hallucination: {json_string[:50]}...")
                        
                    except Exception as e:
                        print(f"Unexpected Error on email {i} (Attempt {attempt+1}/{max_retries}): {e}")
                
                # If we failed 3 times, write a blank/error row so we don't lose alignment in our CSV
                if not success:
                    print(f"-> Skipping email {i} after {max_retries} failed attempts.")
                    writer.writerow([email['Unnamed: 0'], email['Email Type'], "ERROR", "ERROR", "Failed after max retries"])
    
        print(f"\nAnalysis complete.\nResults saved to '{csv_file}'\n")



In [ ]:
model_list = ['gemma3:4b', 'llama3:latest', 'qwen3.5:4b', 'gemma3:12b']
email_list = sample2
save_folder = "Model Evaluation 3 - System Parameter and Increasing Precision"

test_models(email_list, model_list, save_folder)
evaluate_models(model_list, save_folder)

# Iteration 4: Prompt Engineering to Increase Recall



In [ ]:
import requests
import csv
import json
import os
from tqdm import tqdm


def test_models(email_list, model_list, save_folder):
    api_url = "http://localhost:11434/api/generate"
    os.makedirs(save_folder, exist_ok=True)

    print(f"===Models selected: {model_list}===")

    for model_name in model_list:
        print(f'\nStarting phishing email analysis...\nEmail count: {email_list.num_rows}\nModel: {model_name}\n')
        
        csv_file = os.path.join(save_folder, f"{model_name.replace(':', '_')}.csv")
        with open(csv_file, mode='w', newline='') as file:
            writer = csv.writer(file)
            writer.writerow(["Email", "Label", "is_phishing_verdict", "risk_level", "explanation"])
            
            for i in tqdm(range(email_list.num_rows)):
                email = email_list[i]

                max_retries = 5
                success = False

                for attempt in range(max_retries):
                    payload = {
                        "model": model_name,
                        "system": """
                            You are an expert cybersecurity analyst. Your only task is to analyze an email and determine if it is phishing or safe.
                            
                            CRITICAL RULES:
                            1. ZERO TRUST POLICY: If an email contains ANY suspicious elements (e.g., unexpected links, slight urgency, unusual requests), flag it as phishing. Do not give the sender the benefit of the doubt.
                            2. Flag subtle manipulation: Even if an email looks like standard corporate communication or an automated newsletter, flag it if it attempts to harvest credentials, bypass standard procedures, or manipulate the recipient.
                            3. Do not assume an email is safe just because it lacks an obvious malicious link; pure social engineering attempts must be flagged.
                            4. If ambiguous or you are unsure, default to Phishing ("is_phishing": true).
                            5. You must return a raw JSON object and absolutely nothing else.
                        """,
                        "prompt": f"""
                            You are an expert cybersecurity analyst. Your task is to analyze an email and determine if it is phishing or safe.
                            You must return a raw JSON object and absolutely nothing else.

                            Here are examples of how to classify emails:

                            --- EXAMPLE 1: Safe Email ---
                            Email: "Ok, Iknow this is blatantly OT but I'm beginning to go insane.\nHad an old Dell Dimension XPS sitting in the corner and decided to\nput it to use, I know it was working pre being stuck in the\ncorner, but when I plugged it in, hit the power nothing happened.\nI opened her up and had a look and say nothing much. A little orange\nLED comes on when I plug her in but that's it, after some googling\nI found some reference to re-seating all the parts, but no change.\nThe problem I'm having is that since the power supply is some Dell\nspecific one, ATX block with what looks like one of the old AT\npower connectors, I cant figure out weather this is a Mobo prob\nor a PSU prob. Just to futily try and drag this back OT, I want\nto install Linux on it when I get it working. If anyone knows\nwhat the problem might be give me a shout.Cheers,\nPeter.--\nPeter Aherne, Software Engineer, \nMotorola Ireland Ltd.\nPh: +353 21 4511234 Mobile: +353 87 2246834-- \nIrish Linux Users' Group: ilug@linux.ie\nhttp://www.linux.ie/mailman/listinfo/ilug for (un)subscription information.\nList maintainer: listmaster@linux.ie"
                            Output:
                            {{
                                "is_phishing": false,
                                "risk_level": "Low",
                                "explanation": "Legitimate technical support inquiry sent to a public mailing list. It contains a verifiable signature block with standard contact details, lacks any fabricated urgency or requests for sensitive information, and the included URL is a standard, safe mailing list administrative link."
                            }}

                            --- EXAMPLE 2: Phishing Email ---
                            Email: 'emailing : mon , 30 aug 2004 19 : 28 : 52 - 0800 dear sir / madam ; from our records we understand that you are inquiring about a new profession . we have a limited , ont time offer . our university can offer you a pre - qualified degree in your field of choice . we offer signing bonuses of up to $ 15 , 000 in your profession . to obtain your degree with valid transcripts & information on new career bonusus , follow our link : sincerely , dr . julie paige administration office this communication is of private matter . if you have received this and ar enot the individual or group it may concern or show interest in this please follow so we know http : / / 1 highereducation . com / st . html and the proper measures will proficiently expidited in a timely manner .'
                            Output:
                            {{
                                "is_phishing": true,
                                "risk_level": "High",
                                "explanation": "Classic scam lure. Uses a generic greeting, contains multiple spelling and grammatical errors ('ont time', 'bonusus', 'expidited'), offers highly unrealistic financial incentives for a 'pre-qualified degree', and directs the victim to a suspicious, unofficial domain."
                            }}

                            --- ACTUAL TASK ---
                            Analyse this email for phishing indicators.
                            Email: "{email['Email Text']}"

                            Return a raw JSON object with this exact structure:
                            {{
                                "explanation": "Step-by-step reasoning of the technical and social indicators...",
                                "risk_level": "Low" | "Medium" | "High",
                                "is_phishing": boolean
                            }}
                        """,
                        "format": "json",  # Forces local model into JSON mode
                        "stream": False,
                        "options": {
                            "temperature": 0.0,
                            "seed": 24,
                            "num_ctx": 4096
                        }
                    }

                    try:
                        response = requests.post(api_url, json=payload)
                        response.raise_for_status()
                        
                        raw_response = response.json().get("response", "").strip()
                        thinking_text = response.json().get("thinking", "").strip()

                        if raw_response != "":
                            json_string= raw_response

                        else:
                            json_string = thinking_text

                        verdict = json.loads(json_string)

                        writer.writerow([email['Unnamed: 0'], email['Email Type'], verdict['is_phishing'], verdict.get('risk_level', 'Missing'), verdict.get('explanation', 'Missing')])

                        success = True
                        break  # Exit retry loop on success

                    except requests.exceptions.RequestException as e:
                        print(f"API Error on email {i} (Attempt {attempt+1}/{max_retries}): {e}")
                        
                    except json.JSONDecodeError as e:
                        print(f"JSON Error on email {i} (Attempt {attempt+1}/{max_retries}). Hallucination: {json_string[:50]}...")
                        
                    except Exception as e:
                        print(f"Unexpected Error on email {i} (Attempt {attempt+1}/{max_retries}): {e}")
                
                # If we failed 3 times, write a blank/error row so we don't lose alignment in our CSV
                if not success:
                    print(f"-> Skipping email {i} after {max_retries} failed attempts.")
                    writer.writerow([email['Unnamed: 0'], email['Email Type'], "ERROR", "ERROR", "Failed after max retries"])
    
        print(f"\nAnalysis complete.\nResults saved to '{csv_file}'\n")



In [ ]:
model_list = ['gemma3:4b', 'llama3:latest', 'qwen3.5:4b', 'gemma3:12b']
email_list = sample2
save_folder = "Model Evaluation 4 - Increasing Recall"

test_models(email_list, model_list, save_folder)
evaluate_models(model_list, save_folder)

# Iteration 5: Lowering Temperature to 0.3

In [ ]:
import requests
import csv
import json
import os
from tqdm import tqdm


def test_models(email_list, model_list, save_folder):
    api_url = "http://localhost:11434/api/generate"
    os.makedirs(save_folder, exist_ok=True)

    print(f"===Models selected: {model_list}===")

    for model_name in model_list:
        print(f'\nStarting phishing email analysis...\nEmail count: {email_list.num_rows}\nModel: {model_name}\n')
        
        csv_file = os.path.join(save_folder, f"{model_name.replace(':', '_')}.csv")
        with open(csv_file, mode='w', newline='') as file:
            writer = csv.writer(file)
            writer.writerow(["Email", "Label", "is_phishing_verdict", "risk_level", "explanation"])
            
            for i in tqdm(range(email_list.num_rows)):
                email = email_list[i]

                max_retries = 5
                success = False

                for attempt in range(max_retries):
                    payload = {
                        "model": model_name,
                        "system": """
                            You are an expert cybersecurity analyst. Your only task is to analyze an email and determine if it is phishing or safe.
                            
                            CRITICAL RULES:
                            1. ZERO TRUST POLICY: If an email contains ANY suspicious elements (e.g., unexpected links, slight urgency, unusual requests), flag it as phishing. Do not give the sender the benefit of the doubt.
                            2. Flag subtle manipulation: Even if an email looks like standard corporate communication or an automated newsletter, flag it if it attempts to harvest credentials, bypass standard procedures, or manipulate the recipient.
                            3. Do not assume an email is safe just because it lacks an obvious malicious link; pure social engineering attempts must be flagged.
                            4. If ambiguous or you are unsure, default to Phishing ("is_phishing": true).
                            5. You must return a raw JSON object and absolutely nothing else.
                        """,
                        "prompt": f"""
                            You are an expert cybersecurity analyst. Your task is to analyze an email and determine if it is phishing or safe.
                            You must return a raw JSON object and absolutely nothing else.

                            Here are examples of how to classify emails:

                            --- EXAMPLE 1: Safe Email ---
                            Email: "Ok, Iknow this is blatantly OT but I'm beginning to go insane.\nHad an old Dell Dimension XPS sitting in the corner and decided to\nput it to use, I know it was working pre being stuck in the\ncorner, but when I plugged it in, hit the power nothing happened.\nI opened her up and had a look and say nothing much. A little orange\nLED comes on when I plug her in but that's it, after some googling\nI found some reference to re-seating all the parts, but no change.\nThe problem I'm having is that since the power supply is some Dell\nspecific one, ATX block with what looks like one of the old AT\npower connectors, I cant figure out weather this is a Mobo prob\nor a PSU prob. Just to futily try and drag this back OT, I want\nto install Linux on it when I get it working. If anyone knows\nwhat the problem might be give me a shout.Cheers,\nPeter.--\nPeter Aherne, Software Engineer, \nMotorola Ireland Ltd.\nPh: +353 21 4511234 Mobile: +353 87 2246834-- \nIrish Linux Users' Group: ilug@linux.ie\nhttp://www.linux.ie/mailman/listinfo/ilug for (un)subscription information.\nList maintainer: listmaster@linux.ie"
                            Output:
                            {{
                                "is_phishing": false,
                                "risk_level": "Low",
                                "explanation": "Legitimate technical support inquiry sent to a public mailing list. It contains a verifiable signature block with standard contact details, lacks any fabricated urgency or requests for sensitive information, and the included URL is a standard, safe mailing list administrative link."
                            }}

                            --- EXAMPLE 2: Phishing Email ---
                            Email: 'emailing : mon , 30 aug 2004 19 : 28 : 52 - 0800 dear sir / madam ; from our records we understand that you are inquiring about a new profession . we have a limited , ont time offer . our university can offer you a pre - qualified degree in your field of choice . we offer signing bonuses of up to $ 15 , 000 in your profession . to obtain your degree with valid transcripts & information on new career bonusus , follow our link : sincerely , dr . julie paige administration office this communication is of private matter . if you have received this and ar enot the individual or group it may concern or show interest in this please follow so we know http : / / 1 highereducation . com / st . html and the proper measures will proficiently expidited in a timely manner .'
                            Output:
                            {{
                                "is_phishing": true,
                                "risk_level": "High",
                                "explanation": "Classic scam lure. Uses a generic greeting, contains multiple spelling and grammatical errors ('ont time', 'bonusus', 'expidited'), offers highly unrealistic financial incentives for a 'pre-qualified degree', and directs the victim to a suspicious, unofficial domain."
                            }}

                            --- ACTUAL TASK ---
                            Analyse this email for phishing indicators.
                            Email: "{email['Email Text']}"

                            Return a raw JSON object with this exact structure:
                            {{
                                "explanation": "Step-by-step reasoning of the technical and social indicators...",
                                "risk_level": "Low" | "Medium" | "High",
                                "is_phishing": boolean
                            }}
                        """,
                        "format": "json",  # Forces local model into JSON mode
                        "stream": False,
                        "options": {
                            "temperature": 0.3
                        }
                    }

                    try:
                        response = requests.post(api_url, json=payload)
                        response.raise_for_status()
                        
                        raw_response = response.json().get("response", "").strip()
                        thinking_text = response.json().get("thinking", "").strip()

                        if raw_response != "":
                            json_string= raw_response

                        else:
                            json_string = thinking_text

                        verdict = json.loads(json_string)

                        writer.writerow([email['Unnamed: 0'], email['Email Type'], verdict['is_phishing'], verdict.get('risk_level', 'Missing'), verdict.get('explanation', 'Missing')])

                        success = True
                        break  # Exit retry loop on success

                    except requests.exceptions.RequestException as e:
                        print(f"API Error on email {i} (Attempt {attempt+1}/{max_retries}): {e}")
                        
                    except json.JSONDecodeError as e:
                        print(f"JSON Error on email {i} (Attempt {attempt+1}/{max_retries}). Hallucination: {json_string[:50]}...")
                        
                    except Exception as e:
                        print(f"Unexpected Error on email {i} (Attempt {attempt+1}/{max_retries}): {e}")
                
                # If we failed 3 times, write a blank/error row so we don't lose alignment in our CSV
                if not success:
                    print(f"-> Skipping email {i} after {max_retries} failed attempts.")
                    writer.writerow([email['Unnamed: 0'], email['Email Type'], "ERROR", "ERROR", "Failed after max retries"])
    
        print(f"\nAnalysis complete.\nResults saved to '{csv_file}'\n")



In [ ]:
model_list = ['gemma3:4b', 'llama3:latest', 'qwen3.5:4b', 'gemma3:12b']
email_list = sample2
save_folder = "Model Evaluation 5 - Decreasing Temperature"

test_models(email_list, model_list, save_folder)
evaluate_models(model_list, save_folder)

# Iteration 6: Five-Shot Prompting with Hard Negative Mining

## Finding dead spots/ blind spots and identifying patterns

In [ ]:
# import os
# import csv

# def search_dead_spots(folder_list, model_list):
#     # Dictionary to keep track of failures per email ID
#     # Format: { 'email_id': {'count': 0, 'is_phishing': bool, 'risk_counts': {'High': 2, 'Low': 1}} }
#     failure_tracker = {}
    
#     for folder in folder_list:
#         for model_name in model_list:
#             safe_model_name = model_name.replace(':', '_')
#             file_path = os.path.join(folder, f"{safe_model_name}.csv")
            
#             if not os.path.exists(file_path):
#                 print(f"Skipping missing file: {file_path}")
#                 continue
                
#             with open(file_path, mode='r', encoding='utf-8', errors='replace') as file:
#                 reader = csv.DictReader(file)
                
#                 for row in reader:
#                     email_id = row['Email']
#                     label = str(row['Label']).strip().lower()
#                     verdict = str(row['is_phishing_verdict']).strip().lower()
                    
#                     # Grab the risk level and normalize it (e.g., 'high', 'HIGH' -> 'High')
#                     risk_level = str(row['risk_level']).strip().title()
                    
#                     if verdict == 'error':
#                         continue
                        
#                     target_phishing_label = '0' 
                    
#                     is_actual_phishing = (label == target_phishing_label)
#                     predicted_phishing = (verdict == 'true')
                    
#                     # Check if the model got it wrong
#                     if is_actual_phishing != predicted_phishing:
#                         if email_id not in failure_tracker:
#                             failure_tracker[email_id] = {
#                                 'count': 0, 
#                                 'is_phishing': is_actual_phishing,
#                                 'risk_counts': {} # New dictionary to tally risk levels
#                             }
                        
#                         # Increment the overall failure count
#                         failure_tracker[email_id]['count'] += 1
                        
#                         # Increment the specific risk level count for this email
#                         if risk_level not in failure_tracker[email_id]['risk_counts']:
#                             failure_tracker[email_id]['risk_counts'][risk_level] = 0
#                         failure_tracker[email_id]['risk_counts'][risk_level] += 1

#     # Convert tracker dictionary to the requested list of dictionaries
#     dead_spots = []
#     for email_id, data in failure_tracker.items():
#         # Find the most frequent risk level by checking which key has the highest value
#         risk_counts = data['risk_counts']
#         if risk_counts:
#             majority_risk = max(risk_counts, key=risk_counts.get)
#         else:
#             majority_risk = "Unknown"

#         dead_spots.append({
#             'email': email_id,
#             'misidentified_no': data['count'],
#             'is_phishing': data['is_phishing'],
#             'majority_risk_level': majority_risk
#         })
        
#     # Sort the list in descending order by the number of times it was misidentified
#     dead_spots.sort(key=lambda x: x['misidentified_no'], reverse=True)
    
#     # Slice the top 50
#     top_50_dead_spots = dead_spots[:50]
    
#     # Print the results nicely
#     print(f"\n=== Top {len(top_50_dead_spots)} Dead Spots ===")
#     for spot in top_50_dead_spots:
#         print(spot)
        
#     return top_50_dead_spots


# folders = [
#     "Model Evaluation 1 - Initial Stage", 
#     "Model Evaluation 2 - Two-Shot Prompting", 
#     "Model Evaluation 3 - System Parameter and Increasing Precision", 
#     "Model Evaluation 4 - Increasing Recall", 
#     "Model Evaluation 5 - Decreasing Temperature"
# ]
# models = ["gemma3:4b", "qwen3.5:4b", "llama3_latest", "gemma3:12b"]

# dead_spots = search_dead_spots(folders, models)
# print(dead_spots)

In [ ]:
# def get_email_by_id(email_list, target_id):
#     """Searches the dataset for a specific email ID."""
    
#     for email in email_list:
#         # We convert both to strings just in case the CSV read the ID as text 
#         # but the original dataset stored it as an integer.
#         if str(email['Unnamed: 0']) == str(target_id):
#             return email
            
#     return None  # Returns None if the email somehow isn't found

# email = get_email_by_id(sample1, 16030)
# print(email)  # Replace 12345 with the actual email ID you want to retrieve
# print(len(email['Email Text']))

In [ ]:
# i = 1
# for x in dead_spots:
#     email = get_email_by_id(sample1, x['email'])
#     print(f"Email {i}:")
#     print(email)
#     print(len(email['Email Text']))
#     print("")
#     i += 1

In [ ]:
# split2 = split1['train'].train_test_split(test_size=100, stratify_by_column='Email Type', seed=20)
# sample3 = split2['test']

# for email in sample3:
#     if email['Email Text'] is None:
#         continue
#     if len(email['Email Text']) < 200:
#         print(email)
#         print(len(email['Email Text']))
#         print("")

## Prompt Execution

In [ ]:
import requests
import csv
import json
import os
from tqdm import tqdm


def test_models(email_list, model_list, save_folder):
    api_url = "http://localhost:11434/api/generate"
    os.makedirs(save_folder, exist_ok=True)

    print(f"===Models selected: {model_list}===")

    for model_name in model_list:
        print(f'\nStarting phishing email analysis...\nEmail count: {email_list.num_rows}\nModel: {model_name}\n')
        
        csv_file = os.path.join(save_folder, f"{model_name.replace(':', '_')}.csv")
        with open(csv_file, mode='w', newline='') as file:
            writer = csv.writer(file)
            writer.writerow(["Email", "Label", "is_phishing_verdict", "risk_level", "explanation"])
            
            for i in tqdm(range(email_list.num_rows)):
                email = email_list[i]

                max_retries = 5
                success = False

                for attempt in range(max_retries):
                    payload = {
                        "model": model_name,
                        "system": """
                            You are an expert cybersecurity analyst. Your only task is to analyze an email and determine if it is phishing or safe.
                            
                            CRITICAL RULES:
                            1. ZERO TRUST POLICY: If an email contains ANY suspicious elements (e.g., unexpected links, slight urgency, unusual requests), flag it as phishing. Do not give the sender the benefit of the doubt.
                            2. Flag subtle manipulation: Even if an email looks like standard corporate communication or an automated newsletter, flag it if it attempts to harvest credentials, bypass standard procedures, or manipulate the recipient.
                            3. Do not assume an email is safe just because it lacks an obvious malicious link; pure social engineering attempts must be flagged.
                            4. If ambiguous or you are unsure, default to Phishing ("is_phishing": true).
                            5. You must return a raw JSON object and absolutely nothing else.
                        """,
                        "prompt": f"""
                            Here are examples of how to classify emails:

                            --- EXAMPLE 1: Safe Email ---
                            Email: "Ok, Iknow this is blatantly OT but I'm beginning to go insane.\nHad an old Dell Dimension XPS sitting in the corner and decided to\nput it to use, I know it was working pre being stuck in the\ncorner, but when I plugged it in, hit the power nothing happened.\nI opened her up and had a look and say nothing much. A little orange\nLED comes on when I plug her in but that's it, after some googling\nI found some reference to re-seating all the parts, but no change.\nThe problem I'm having is that since the power supply is some Dell\nspecific one, ATX block with what looks like one of the old AT\npower connectors, I cant figure out weather this is a Mobo prob\nor a PSU prob. Just to futily try and drag this back OT, I want\nto install Linux on it when I get it working. If anyone knows\nwhat the problem might be give me a shout.Cheers,\nPeter.--\nPeter Aherne, Software Engineer, \nMotorola Ireland Ltd.\nPh: +353 21 4511234 Mobile: +353 87 2246834-- \nIrish Linux Users' Group: ilug@linux.ie\nhttp://www.linux.ie/mailman/listinfo/ilug for (un)subscription information.\nList maintainer: listmaster@linux.ie"
                            Output:
                            {{
                                "explanation": "Technical support inquiry sent to a public mailing list. It contains a verifiable signature block with standard contact details, lacks any fabricated urgency or requests for sensitive information, and the included URL is a standard, safe mailing list administrative link. While the email has some spelling errors, it does not contain any of the common indicators of phishing such as suspicious links, requests for sensitive information, or unrealistic offers. The tone and content are consistent with a legitimate technical support request.",
                                "risk_level": "Low",
                                "is_phishing": false
                            }}

                            --- EXAMPLE 2: Phishing Email ---
                            Email: 'emailing : mon , 30 aug 2004 19 : 28 : 52 - 0800 dear sir / madam ; from our records we understand that you are inquiring about a new profession . we have a limited , ont time offer . our university can offer you a pre - qualified degree in your field of choice . we offer signing bonuses of up to $ 15 , 000 in your profession . to obtain your degree with valid transcripts & information on new career bonusus , follow our link : sincerely , dr . julie paige administration office this communication is of private matter . if you have received this and ar enot the individual or group it may concern or show interest in this please follow so we know http : / / 1 highereducation . com / st . html and the proper measures will proficiently expidited in a timely manner .'
                            Output:
                            {{
                                "explanation": "Uses a generic greeting, contains multiple spelling and grammatical errors ('ont time', 'bonusus', 'expidited'), offers highly big unrealistic financial incentive ('$ 15 , 000') for a 'pre-qualified degree'. Too good to be true. Directs the victim to a suspicious, unofficial domain. The email also mentions 'limited on time offer' which is a common tactic to create urgency and pressure the recipient into acting without due consideration. All of these are strong indicators of a phishing attempt.",
                                "risk_level": "High",
                                "is_phishing": true
                            }}

                            --- EXAMPLE 3: Phishing Email ---
                            Email: "returned mail : see transcript for details the original message was received at tue , 19 jul 2005 07 : 06 : 09 - 0400 from root @ localhost - - - - - the following addresses had permanent fatal errors - - - - - antique ( reason : can \' t create ( user ) output file ) ( expanded from : ) - - - - - transcript of session follows - - - - - procmail : quota exceeded while writing " / var / spool / mail / antique " 550 5 . 0 . 0 antique . . . can \' t create output"
                            Output:
                            {{
                                "explanation": "This is a classic 'Fake NDR' (Non-Delivery Report) or 'Quota Exceeded' spoof. The email mimics an automated server error ('fatal errors', 'quota exceeded') to create sudden panic or confusion. Phishers frequently spoof these administrative bounce messages to trick victims into clicking a malicious link to 'upgrade their quota' or 'view the undelivered message', bypassing the victim's natural skepticism of external emails.",
                                "risk_level": "High",
                                "is_phishing": true
                            }}

                            --- EXAMPLE 4: Safe Email ---
                            Email: "please grant access : application request ( kker - 4 r 3 klb ) security resource line item request kker - 4 r 3 klb has been submitted for your processing . to view the request , click your left mouse button on the notes document link below ."
                            Output:
                            {{
                                "explanation": "Appears to be a legitimate, automated internal IT or ticketing system notification. It includes a highly specific, standard-formatted reference tracking code ('kker - 4 r 3 klb') and uses dry, administrative language. It does not threaten negative consequences or create artificial urgency. The instruction to click a 'notes document link' is standard operational practice for legacy enterprise environments (like Lotus Notes).",
                                "risk_level": "Low",
                                "is_phishing": false
                            }}
                            
                            --- EXAMPLE 5: Safe Email ---
                            Email: 'hpl nom for may 11 , 2001 ( see attached file : hplno 511 . xls ) - hplno 511 . xls'
                            Output:
                            {{
                                "explanation": "This is a routine, automated or highly abbreviated B2B transactional email. It uses specific operational shorthand ('hpl nom', indicating a daily business nomination) paired with a specific date. The attached spreadsheet ('hplno 511 . xls') directly correlates with the date (May 11 -> 5/11) and the context of the message. It contains zero social engineering indicators, no artificial urgency, and no requests for credentials, pointing to a standard, safe internal workflow.",
                                "risk_level": "Low",
                                "is_phishing": false
                            }}

                            --- ACTUAL TASK ---
                            Analyse this email for phishing indicators.
                            Email: "{email['Email Text']}"

                            Return a raw JSON object with this exact structure:
                            {{
                                "explanation": "Step-by-step reasoning of the technical and social indicators...",
                                "risk_level": "Low" | "Medium" | "High",
                                "is_phishing": boolean
                            }}
                        """,
                        "format": "json",  # Forces local model into JSON mode
                        "stream": False,
                        "options": {
                            "temperature": 0.0,
                            "seed": 24,
                            "num_ctx": 4096
                        }
                    }

                    try:
                        response = requests.post(api_url, json=payload)
                        response.raise_for_status()
                        
                        raw_response = response.json().get("response", "").strip()
                        thinking_text = response.json().get("thinking", "").strip()

                        full_text = raw_response if raw_response else thinking_text

                        start_idx = full_text.find('{')
                        end_idx = full_text.rfind('}')
                        
                        # If both brackets exist and are in the correct order, slice the string
                        if start_idx != -1 and end_idx != -1 and end_idx > start_idx:
                            json_string = full_text[start_idx:end_idx + 1]
                        else:
                            # Force an error to trigger the retry loop if no brackets are found
                            raise ValueError("No JSON brackets found in the AI response.")

                        verdict = json.loads(json_string)

                        writer.writerow([email['Unnamed: 0'], email['Email Type'], verdict['is_phishing'], verdict.get('risk_level', 'Missing'), verdict.get('explanation', 'Missing')])

                        success = True
                        break  # Exit retry loop on success

                    except requests.exceptions.RequestException as e:
                        print(f"API Error on email {i} (Attempt {attempt+1}/{max_retries}): {e}")
                        
                    except json.JSONDecodeError as e:
                        print(f"JSON Error on email {i} (Attempt {attempt+1}/{max_retries}). Hallucination: {json_string[:50]}...")
                        
                    except Exception as e:
                        print(f"Unexpected Error on email {i} (Attempt {attempt+1}/{max_retries}): {e}")
                
                # If we failed 5 times, write a blank/error row so we don't lose alignment in our CSV
                if not success:
                    print(f"-> Skipping email {i} after {max_retries} failed attempts.")
                    writer.writerow([email['Unnamed: 0'], email['Email Type'], "ERROR", "ERROR", "Failed after max retries"])
    
        print(f"\nAnalysis complete.\nResults saved to '{csv_file}'\n")



In [ ]:
model_list = ['gemma3:4b', 'llama3:latest', 'qwen3.5:4b', 'gemma3:12b', 'mixtral:8x7b']
email_list = sample2
save_folder = "Model Evaluation 6 - Five-Shot Prompting to Target Dead Spots"

test_models(email_list, model_list, save_folder)
evaluate_models(model_list, save_folder)

# Iteration 7: Self-Consistency

In [34]:
import os
import csv
import json
import requests
from tqdm import tqdm

def test_models(email_list, model_list, save_folder):
    api_url = "http://localhost:11434/api/generate"
    os.makedirs(save_folder, exist_ok=True)

    print(f"===Models selected: {model_list}===")

    for model_name in model_list:
        print(f'\nStarting Self-Consistency analysis...\nEmail count: {email_list.num_rows}\nModel: {model_name}\n')
        
        csv_file = os.path.join(save_folder, f"{model_name.replace(':', '_')}.csv")
        with open(csv_file, mode='w', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow(["Email", "Label", "is_phishing_verdict", "risk_level", "explanation"])
            
            for i in tqdm(range(email_list.num_rows)):
                email = email_list[i]
                
                # --- SELF-CONSISTENCY SETTINGS ---
                num_thought_paths = 3
                collected_verdicts = []
                
                # Run the exact same email through the model multiple times
                for path in range(num_thought_paths):
                    max_retries = 5
                    success = False

                    for attempt in range(max_retries):
                        payload = {
                            "model": model_name,
                            "system": """
                                You are an expert cybersecurity analyst. Your only task is to analyze an email and determine if it is phishing or safe.
                                
                                CRITICAL RULES:
                                1. ZERO TRUST POLICY: If an email contains ANY suspicious elements (e.g., unexpected links, slight urgency, unusual requests), flag it as phishing. Do not give the sender the benefit of the doubt.
                                2. Flag subtle manipulation: Even if an email looks like standard corporate communication or an automated newsletter, flag it if it attempts to harvest credentials, bypass standard procedures, or manipulate the recipient.
                                3. Do not assume an email is safe just because it lacks an obvious malicious link; pure social engineering attempts must be flagged.
                                4. If ambiguous or you are unsure, default to Phishing ("is_phishing": true).
                                5. You must return a raw JSON object and absolutely nothing else.
                            """,
                            "prompt": f"""
                                Here are examples of how to classify emails:

                                --- EXAMPLE 1: Safe Email ---
                                Email: "Ok, Iknow this is blatantly OT but I'm beginning to go insane.\nHad an old Dell Dimension XPS sitting in the corner and decided to\nput it to use, I know it was working pre being stuck in the\ncorner, but when I plugged it in, hit the power nothing happened.\nI opened her up and had a look and say nothing much. A little orange\nLED comes on when I plug her in but that's it, after some googling\nI found some reference to re-seating all the parts, but no change.\nThe problem I'm having is that since the power supply is some Dell\nspecific one, ATX block with what looks like one of the old AT\npower connectors, I cant figure out weather this is a Mobo prob\nor a PSU prob. Just to futily try and drag this back OT, I want\nto install Linux on it when I get it working. If anyone knows\nwhat the problem might be give me a shout.Cheers,\nPeter.--\nPeter Aherne, Software Engineer, \nMotorola Ireland Ltd.\nPh: +353 21 4511234 Mobile: +353 87 2246834-- \nIrish Linux Users' Group: ilug@linux.ie\nhttp://www.linux.ie/mailman/listinfo/ilug for (un)subscription information.\nList maintainer: listmaster@linux.ie"
                                Output:
                                {{
                                    "explanation": "Technical support inquiry sent to a public mailing list. It contains a verifiable signature block with standard contact details, lacks any fabricated urgency or requests for sensitive information, and the included URL is a standard, safe mailing list administrative link. While the email has some spelling errors, it does not contain any of the common indicators of phishing such as suspicious links, requests for sensitive information, or unrealistic offers. The tone and content are consistent with a legitimate technical support request.",
                                    "risk_level": "Low",
                                    "is_phishing": false
                                }}

                                --- EXAMPLE 2: Phishing Email ---
                                Email: 'emailing : mon , 30 aug 2004 19 : 28 : 52 - 0800 dear sir / madam ; from our records we understand that you are inquiring about a new profession . we have a limited , ont time offer . our university can offer you a pre - qualified degree in your field of choice . we offer signing bonuses of up to $ 15 , 000 in your profession . to obtain your degree with valid transcripts & information on new career bonusus , follow our link : sincerely , dr . julie paige administration office this communication is of private matter . if you have received this and ar enot the individual or group it may concern or show interest in this please follow so we know http : / / 1 highereducation . com / st . html and the proper measures will proficiently expidited in a timely manner .'
                                Output:
                                {{
                                    "explanation": "Uses a generic greeting, contains multiple spelling and grammatical errors ('ont time', 'bonusus', 'expidited'), offers highly big unrealistic financial incentive ('$ 15 , 000') for a 'pre-qualified degree'. Too good to be true. Directs the victim to a suspicious, unofficial domain. The email also mentions 'limited on time offer' which is a common tactic to create urgency and pressure the recipient into acting without due consideration. All of these are strong indicators of a phishing attempt.",
                                    "risk_level": "High",
                                    "is_phishing": true
                                }}

                                --- EXAMPLE 3: Phishing Email ---
                                Email: "returned mail : see transcript for details the original message was received at tue , 19 jul 2005 07 : 06 : 09 - 0400 from root @ localhost - - - - - the following addresses had permanent fatal errors - - - - - antique ( reason : can \' t create ( user ) output file ) ( expanded from : ) - - - - - transcript of session follows - - - - - procmail : quota exceeded while writing " / var / spool / mail / antique " 550 5 . 0 . 0 antique . . . can \' t create output"
                                Output:
                                {{
                                    "explanation": "This is a classic 'Fake NDR' (Non-Delivery Report) or 'Quota Exceeded' spoof. The email mimics an automated server error ('fatal errors', 'quota exceeded') to create sudden panic or confusion. Phishers frequently spoof these administrative bounce messages to trick victims into clicking a malicious link to 'upgrade their quota' or 'view the undelivered message', bypassing the victim's natural skepticism of external emails.",
                                    "risk_level": "High",
                                    "is_phishing": true
                                }}

                                --- EXAMPLE 4: Safe Email ---
                                Email: "please grant access : application request ( kker - 4 r 3 klb ) security resource line item request kker - 4 r 3 klb has been submitted for your processing . to view the request , click your left mouse button on the notes document link below ."
                                Output:
                                {{
                                    "explanation": "Appears to be a legitimate, automated internal IT or ticketing system notification. It includes a highly specific, standard-formatted reference tracking code ('kker - 4 r 3 klb') and uses dry, administrative language. It does not threaten negative consequences or create artificial urgency. The instruction to click a 'notes document link' is standard operational practice for legacy enterprise environments (like Lotus Notes).",
                                    "risk_level": "Low",
                                    "is_phishing": false
                                }}
                                
                                --- EXAMPLE 5: Safe Email ---
                                Email: 'hpl nom for may 11 , 2001 ( see attached file : hplno 511 . xls ) - hplno 511 . xls'
                                Output:
                                {{
                                    "explanation": "This is a routine, automated or highly abbreviated B2B transactional email. It uses specific operational shorthand ('hpl nom', indicating a daily business nomination) paired with a specific date. The attached spreadsheet ('hplno 511 . xls') directly correlates with the date (May 11 -> 5/11) and the context of the message. It contains zero social engineering indicators, no artificial urgency, and no requests for credentials, pointing to a standard, safe internal workflow.",
                                    "risk_level": "Low",
                                    "is_phishing": false
                                }}

                                --- ACTUAL TASK ---
                                Analyse this email for phishing indicators.
                                Email: "{email['Email Text']}"

                                Return a raw JSON object with this exact structure:
                                {{
                                    "explanation": "Step-by-step reasoning of the technical and social indicators...",
                                    "risk_level": "Low" | "Medium" | "High",
                                    "is_phishing": boolean
                                }}
                            """,
                            "format": "json",
                            "stream": False,
                            "options": {
                                "temperature": 0.4, # INCREASED: Allows the model to explore different thought paths
                                "num_ctx": 4096
                            }
                        }

                        try:
                            response = requests.post(api_url, json=payload)
                            response.raise_for_status()
                            
                            raw_response = response.json().get("response", "").strip()
                            thinking_text = response.json().get("thinking", "").strip()

                            full_text = raw_response if raw_response else thinking_text

                            start_idx = full_text.find('{')
                            end_idx = full_text.rfind('}')
                            
                            if start_idx != -1 and end_idx != -1 and end_idx > start_idx:
                                json_string = full_text[start_idx:end_idx + 1]
                            else:
                                raise ValueError("No JSON brackets found in the AI response.")

                            verdict = json.loads(json_string)

                            # Save this successful path's result to our temporary list
                            collected_verdicts.append(verdict)
                            success = True
                            break  # Exit retry loop for this specific path

                        except requests.exceptions.RequestException as e:
                            print(f"API Error on email {i}, path {path+1} (Attempt {attempt+1}/{max_retries}): {e}")
                        except json.JSONDecodeError as e:
                            print(f"JSON Error on email {i}, path {path+1} (Attempt {attempt+1}/{max_retries}).")
                        except Exception as e:
                            print(f"Unexpected Error on email {i}, path {path+1} (Attempt {attempt+1}/{max_retries}): {e}")
                
                    if not success:
                        print(f"-> Path {path+1} for email {i} completely failed.")
                
                # --- SYNTHESIS STAGE: Tally the votes ---
                if len(collected_verdicts) == 0:
                    # Absolute worst-case scenario: all 3 paths failed 5 times each
                    writer.writerow([email['Unnamed: 0'], email['Email Type'], "ERROR", "ERROR", "Failed all paths"])
                else:
                    # Calculate the majority
                    phishing_votes = sum(1 for v in collected_verdicts if v.get('is_phishing') is True)
                    safe_votes = len(collected_verdicts) - phishing_votes
                    
                    # Explicit Tie-Breaking Logic
                    if phishing_votes > safe_votes:
                        majority_is_phishing = True
                    elif safe_votes > phishing_votes:
                        majority_is_phishing = False
                    else:
                        # IT IS A TIE (e.g., 1 to 1). Enforce Zero Trust Policy.
                        majority_is_phishing = True
                    
                    # Find the first dictionary in our list that voted for the winning side
                    winning_response = next(v for v in collected_verdicts if v.get('is_phishing') == majority_is_phishing)
                    
                    # Log the vote count dynamically in the explanation so you can see it in your CSV!
                    vote_tally_note = f"[Vote: {phishing_votes} Phish / {safe_votes} Safe] "
                    final_explanation = vote_tally_note + winning_response.get('explanation', 'Missing explanation')

                    # Write the final, synthesized result to the CSV
                    writer.writerow([
                        email['Unnamed: 0'], 
                        email['Email Type'], 
                        majority_is_phishing, 
                        winning_response.get('risk_level', 'Missing'), 
                        final_explanation
                    ])
                    
    print(f"\nAnalysis complete.\nResults saved to '{save_folder}'\n")

In [35]:
model_list = ['gemma3:4b', 'llama3:latest', 'qwen3.5:4b', 'gemma3:12b']
email_list = sample2
save_folder = "Model Evaluation 7 - Self-Consistency with 3 Thought Paths"

test_models(email_list, model_list, save_folder)
evaluate_models(model_list, save_folder)

===Models selected: ['gemma3:4b', 'llama3:latest', 'qwen3.5:4b', 'gemma3:12b']===

Starting Self-Consistency analysis...
Email count: 2
Model: gemma3:4b



100%|██████████| 2/2 [06:48<00:00, 204.46s/it]



Starting Self-Consistency analysis...
Email count: 2
Model: llama3:latest



100%|██████████| 2/2 [08:30<00:00, 255.26s/it]



Starting Self-Consistency analysis...
Email count: 2
Model: qwen3.5:4b



100%|██████████| 2/2 [10:48<00:00, 324.01s/it]



Starting Self-Consistency analysis...
Email count: 2
Model: gemma3:12b



100%|██████████| 2/2 [20:51<00:00, 625.92s/it]


Analysis complete.
Results saved to 'Model Evaluation 7 - Self-Consistency with 3 Thought Paths'

===Evaluating models: ['gemma3:4b', 'llama3:latest', 'qwen3.5:4b', 'gemma3:12b']===

Model: gemma3:4b. Results evaluated: 2.

	Accuracy: 1.0
	Precision: 1.0
	Recall: 1.0
	F1 Score: 1.0

Model: llama3:latest. Results evaluated: 2.

	Accuracy: 1.0
	Precision: 1.0
	Recall: 1.0
	F1 Score: 1.0

Model: qwen3.5:4b. Results evaluated: 2.

	Accuracy: 1.0
	Precision: 1.0
	Recall: 1.0
	F1 Score: 1.0

Model: gemma3:12b. Results evaluated: 2.

	Accuracy: 1.0
	Precision: 1.0
	Recall: 1.0
	F1 Score: 1.0

===Model evaluation complete. Comparison saved to 'Model Evaluation 7 - Self-Consistency with 3 Thought Paths\model_comparison.csv'===


# Iteration 8: Self-Consistency with Multi-Perspective Prompting

In [ ]:
import os
import csv
import json
import requests
from tqdm import tqdm

def test_models(email_list, model_list, save_folder):
    api_url = "http://localhost:11434/api/generate"
    os.makedirs(save_folder, exist_ok=True)

    print(f"===Models selected: {model_list}===")

    for model_name in model_list:
        print(f'\nStarting Multi-Perspective analysis...\nEmail count: {email_list.num_rows}\nModel: {model_name}\n')
        
        csv_file = os.path.join(save_folder, f"{model_name.replace(':', '_')}.csv")
        with open(csv_file, mode='w', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow(["Email", "Label", "is_phishing_verdict", "risk_level", "explanation"])
            
            for i in tqdm(range(email_list.num_rows)):
                email = email_list[i]
                
                # --- MULTI-PERSPECTIVE SETTINGS ---
                # Define the 3 specific "hats" the model must wear
                perspectives = [
                    {
                        "role_name": "Network/Technical Analyst",
                        "instruction": "Focus heavily on technical indicators: scrutinize URLs, look for spoofed domains, weird routing data, or unusual file attachment types."
                    },
                    {
                        "role_name": "Social Engineering Expert",
                        "instruction": "Focus heavily on psychological manipulation: look for artificial urgency, authoritative pressure, generic greetings, or emotional triggers designed to bypass logic."
                    },
                    {
                        "role_name": "Fraud/Compliance Investigator",
                        "instruction": "Focus heavily on objective risk: look for requests for sensitive credentials, unusual financial transactions, or attempts to bypass standard corporate verification procedures."
                    }
                ]
                
                collected_verdicts = []
                
                # Run the email through the model 3 times, using a different perspective each time
                for perspective in perspectives:
                    max_retries = 5
                    success = False

                    for attempt in range(max_retries):
                        payload = {
                            "model": model_name,
                            "system": f"""
                                You are an expert cybersecurity analyst. Your only task is to analyze an email and determine if it is phishing or safe.
                                
                                CURRENT ROLE CONSTRAINT: You must analyze this email strictly from the perspective of a {perspective['role_name']}. 
                                {perspective['instruction']}
                                
                                CRITICAL RULES:
                                1. ZERO TRUST POLICY: If an email contains ANY suspicious elements (e.g., unexpected links, slight urgency, unusual requests), flag it as phishing. Do not give the sender the benefit of the doubt.
                                2. Flag subtle manipulation: Even if an email looks like standard corporate communication or an automated newsletter, flag it if it attempts to harvest credentials, bypass standard procedures, or manipulate the recipient.
                                3. Do not assume an email is safe just because it lacks an obvious malicious link; pure social engineering attempts must be flagged.
                                4. If ambiguous or you are unsure, default to Phishing ("is_phishing": true).
                                5. You must return a raw JSON object and absolutely nothing else.
                            """,
                            "prompt": f"""
                                Here are examples of how to classify emails:

                                --- EXAMPLE 1: Safe Email ---
                                Email: "Ok, Iknow this is blatantly OT but I'm beginning to go insane.\nHad an old Dell Dimension XPS sitting in the corner and decided to\nput it to use, I know it was working pre being stuck in the\ncorner, but when I plugged it in, hit the power nothing happened.\nI opened her up and had a look and say nothing much. A little orange\nLED comes on when I plug her in but that's it, after some googling\nI found some reference to re-seating all the parts, but no change.\nThe problem I'm having is that since the power supply is some Dell\nspecific one, ATX block with what looks like one of the old AT\npower connectors, I cant figure out weather this is a Mobo prob\nor a PSU prob. Just to futily try and drag this back OT, I want\nto install Linux on it when I get it working. If anyone knows\nwhat the problem might be give me a shout.Cheers,\nPeter.--\nPeter Aherne, Software Engineer, \nMotorola Ireland Ltd.\nPh: +353 21 4511234 Mobile: +353 87 2246834-- \nIrish Linux Users' Group: ilug@linux.ie\nhttp://www.linux.ie/mailman/listinfo/ilug for (un)subscription information.\nList maintainer: listmaster@linux.ie"
                                Output:
                                {{
                                    "explanation": "Technical support inquiry sent to a public mailing list. It contains a verifiable signature block with standard contact details, lacks any fabricated urgency or requests for sensitive information, and the included URL is a standard, safe mailing list administrative link. While the email has some spelling errors, it does not contain any of the common indicators of phishing such as suspicious links, requests for sensitive information, or unrealistic offers. The tone and content are consistent with a legitimate technical support request.",
                                    "risk_level": "Low",
                                    "is_phishing": false
                                }}

                                --- EXAMPLE 2: Phishing Email ---
                                Email: 'emailing : mon , 30 aug 2004 19 : 28 : 52 - 0800 dear sir / madam ; from our records we understand that you are inquiring about a new profession . we have a limited , ont time offer . our university can offer you a pre - qualified degree in your field of choice . we offer signing bonuses of up to $ 15 , 000 in your profession . to obtain your degree with valid transcripts & information on new career bonusus , follow our link : sincerely , dr . julie paige administration office this communication is of private matter . if you have received this and ar enot the individual or group it may concern or show interest in this please follow so we know http : / / 1 highereducation . com / st . html and the proper measures will proficiently expidited in a timely manner .'
                                Output:
                                {{
                                    "explanation": "Uses a generic greeting, contains multiple spelling and grammatical errors ('ont time', 'bonusus', 'expidited'), offers highly big unrealistic financial incentive ('$ 15 , 000') for a 'pre-qualified degree'. Too good to be true. Directs the victim to a suspicious, unofficial domain. The email also mentions 'limited on time offer' which is a common tactic to create urgency and pressure the recipient into acting without due consideration. All of these are strong indicators of a phishing attempt.",
                                    "risk_level": "High",
                                    "is_phishing": true
                                }}

                                --- EXAMPLE 3: Phishing Email ---
                                Email: "returned mail : see transcript for details the original message was received at tue , 19 jul 2005 07 : 06 : 09 - 0400 from root @ localhost - - - - - the following addresses had permanent fatal errors - - - - - antique ( reason : can \' t create ( user ) output file ) ( expanded from : ) - - - - - transcript of session follows - - - - - procmail : quota exceeded while writing " / var / spool / mail / antique " 550 5 . 0 . 0 antique . . . can \' t create output"
                                Output:
                                {{
                                    "explanation": "This is a classic 'Fake NDR' (Non-Delivery Report) or 'Quota Exceeded' spoof. The email mimics an automated server error ('fatal errors', 'quota exceeded') to create sudden panic or confusion. Phishers frequently spoof these administrative bounce messages to trick victims into clicking a malicious link to 'upgrade their quota' or 'view the undelivered message', bypassing the victim's natural skepticism of external emails.",
                                    "risk_level": "High",
                                    "is_phishing": true
                                }}

                                --- EXAMPLE 4: Safe Email ---
                                Email: "please grant access : application request ( kker - 4 r 3 klb ) security resource line item request kker - 4 r 3 klb has been submitted for your processing . to view the request , click your left mouse button on the notes document link below ."
                                Output:
                                {{
                                    "explanation": "Appears to be a legitimate, automated internal IT or ticketing system notification. It includes a highly specific, standard-formatted reference tracking code ('kker - 4 r 3 klb') and uses dry, administrative language. It does not threaten negative consequences or create artificial urgency. The instruction to click a 'notes document link' is standard operational practice for legacy enterprise environments (like Lotus Notes).",
                                    "risk_level": "Low",
                                    "is_phishing": false
                                }}
                                
                                --- EXAMPLE 5: Safe Email ---
                                Email: 'hpl nom for may 11 , 2001 ( see attached file : hplno 511 . xls ) - hplno 511 . xls'
                                Output:
                                {{
                                    "explanation": "This is a routine, automated or highly abbreviated B2B transactional email. It uses specific operational shorthand ('hpl nom', indicating a daily business nomination) paired with a specific date. The attached spreadsheet ('hplno 511 . xls') directly correlates with the date (May 11 -> 5/11) and the context of the message. It contains zero social engineering indicators, no artificial urgency, and no requests for credentials, pointing to a standard, safe internal workflow.",
                                    "risk_level": "Low",
                                    "is_phishing": false
                                }}

                                --- ACTUAL TASK ---
                                Analyse this email for phishing indicators.
                                Email: "{email['Email Text']}"

                                CRITICAL FINAL INSTRUCTION: You MUST return a valid JSON object. Do not output any conversational text. Use this exact structure:
                                {{
                                    "explanation": "Step-by-step reasoning of the technical and social indicators...",
                                    "risk_level": "Low" | "Medium" | "High",
                                    "is_phishing": boolean
                                }}
                            """,
                            "format": "json",
                            "stream": False,
                            "options": {
                                "temperature": 0.2,
                                "num_ctx": 4096
                            }
                        }

                        try:
                            response = requests.post(api_url, json=payload)
                            response.raise_for_status()
                            
                            raw_response = response.json().get("response", "").strip()
                            thinking_text = response.json().get("thinking", "").strip()

                            full_text = raw_response if raw_response else thinking_text

                            start_idx = full_text.find('{')
                            end_idx = full_text.rfind('}')
                            
                            if start_idx != -1 and end_idx != -1 and end_idx > start_idx:
                                json_string = full_text[start_idx:end_idx + 1]
                            else:
                                raise ValueError("No JSON brackets found in the AI response.")

                            verdict = json.loads(json_string)

                            # --- DEBUGGING PRINTS ---
                            raw_val = verdict.get('is_phishing')
                            print(f"\n[DEBUG] Perspective: {perspective['role_name']}")
                            print(f"[DEBUG] Full JSON Dict: {verdict}")
                            print(f"[DEBUG] Value of 'is_phishing': {raw_val}")
                            print(f"[DEBUG] Python Data Type: {type(raw_val)}\n")
                            # ------------------------

                            if "is_phishing" not in verdict:
                                raise ValueError(f"Model returned valid JSON, but forgot the keys! Raw dict: {verdict}")

                            # --- NEW: Tag the verdict with the role that generated it ---
                            verdict['perspective_used'] = perspective['role_name']

                            collected_verdicts.append(verdict)
                            success = True
                            break  

                        except requests.exceptions.RequestException as e:
                            print(f"API Error on email {i}, path {perspective['role_name']} (Attempt {attempt+1}/{max_retries}): {e}")
                        except json.JSONDecodeError as e:
                            print(f"JSON Error on email {i}, path {perspective['role_name']} (Attempt {attempt+1}/{max_retries}).")
                        except Exception as e:
                            print(f"Unexpected Error on email {i}, path {perspective['role_name']} (Attempt {attempt+1}/{max_retries}): {e}")
                
                    if not success:
                        print(f"-> Path {perspective['role_name']} for email {i} completely failed.")
                
                # --- SYNTHESIS STAGE: Tally the votes ---
                if len(collected_verdicts) == 0:
                    writer.writerow([email['Unnamed: 0'], email['Email Type'], "ERROR", "ERROR", "Failed all paths"])
                else:
                    # Calculate the majority
                    phishing_votes = sum(1 for v in collected_verdicts if v.get('is_phishing') is True)
                    safe_votes = len(collected_verdicts) - phishing_votes
                    
                    # Explicit Tie-Breaking Logic
                    if phishing_votes > safe_votes:
                        majority_is_phishing = True
                    elif safe_votes > phishing_votes:
                        majority_is_phishing = False
                    else:
                        # IT IS A TIE (e.g., 1 to 1). Enforce Zero Trust Policy.
                        majority_is_phishing = True
                    
                    # Find the first dictionary in our list that voted for the winning side
                    winning_response = next(v for v in collected_verdicts if v.get('is_phishing') == majority_is_phishing)
                    
                    # Log the vote count AND the specific persona that provided the winning logic!
                    vote_tally_note = f"[Vote: {phishing_votes} Phish / {safe_votes} Safe | Winning Logic: {winning_response.get('perspective_used')}] "
                    final_explanation = vote_tally_note + winning_response.get('explanation', 'Missing explanation')

                    writer.writerow([
                        email['Unnamed: 0'], 
                        email['Email Type'], 
                        majority_is_phishing, 
                        winning_response.get('risk_level', 'Missing'), 
                        final_explanation
                    ])
                    
    print(f"\nAnalysis complete.\nResults saved to '{save_folder}'\n")

In [ ]:
model_list = ['gemma3:4b', 'llama3:latest', 'qwen3.5:4b', 'gemma3:12b']
model_list = ['mixtral:8x7b']
email_list = sample2
save_folder = "Model Evaluation 8 - Self-Consistency with Multi-Perspective Prompting"

test_models(email_list, model_list, save_folder)
evaluate_models(model_list, save_folder)

In [29]:
import os
import csv
import json
import requests
from tqdm import tqdm

def test_models(email_list, model_list, save_folder):
    api_url = "http://localhost:11434/api/generate"
    os.makedirs(save_folder, exist_ok=True)

    print(f"===Models selected: {model_list}===")

    for model_name in model_list:
        print(f'\nStarting Multi-Perspective analysis...\nEmail count: {email_list.num_rows}\nModel: {model_name}\n')
        
        csv_file = os.path.join(save_folder, f"{model_name.replace(':', '_')}.csv")
        with open(csv_file, mode='w', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow(["Email", "Label", "is_phishing_verdict", "risk_level", "explanation"])
            
            for i in tqdm(range(email_list.num_rows)):
                email = email_list[i]
                
                # --- MULTI-PERSPECTIVE SETTINGS ---
                # Define the 3 specific "hats" the model must wear
                perspectives = [
                    {
                        "role_name": "Network/Technical Analyst",
                        "instruction": "Focus heavily on technical indicators: scrutinize URLs, look for spoofed domains, weird routing data, or unusual file attachment types."
                    },
                    {
                        "role_name": "Social Engineering Expert",
                        "instruction": "Focus heavily on psychological manipulation: look for artificial urgency, authoritative pressure, generic greetings, or emotional triggers designed to bypass logic."
                    },
                    {
                        "role_name": "Fraud/Compliance Investigator",
                        "instruction": "Focus heavily on objective risk: look for requests for sensitive credentials, unusual financial transactions, or attempts to bypass standard corporate verification procedures."
                    }
                ]
                
                collected_verdicts = []
                
                # Run the email through the model 3 times, using a different perspective each time
                for perspective in perspectives:
                    max_retries = 5
                    success = False

                    for attempt in range(max_retries):
                        payload = {
                            "model": model_name,
                            "system": f"""
                                You are an expert cybersecurity analyst. Your only task is to analyze an email and determine if it is phishing or safe.
                                
                                CURRENT ROLE CONSTRAINT: You must analyze this email strictly from the perspective of a {perspective['role_name']}. 
                                {perspective['instruction']}
                                
                                CRITICAL RULES:
                                1. ZERO TRUST POLICY: If an email contains ANY suspicious elements (e.g., unexpected links, slight urgency, unusual requests), flag it as phishing. Do not give the sender the benefit of the doubt.
                                2. Flag subtle manipulation: Even if an email looks like standard corporate communication or an automated newsletter, flag it if it attempts to harvest credentials, bypass standard procedures, or manipulate the recipient.
                                3. Do not assume an email is safe just because it lacks an obvious malicious link; pure social engineering attempts must be flagged.
                                4. If ambiguous or you are unsure, default to Phishing ("is_phishing": true).
                                5. You must return a raw JSON object and absolutely nothing else.
                            """,
                            "prompt": f"""
                                Here are examples of how to classify emails:

                                --- EXAMPLE 1: Safe Email ---
                                Email: "Ok, Iknow this is blatantly OT but I'm beginning to go insane.\nHad an old Dell Dimension XPS sitting in the corner and decided to\nput it to use, I know it was working pre being stuck in the\ncorner, but when I plugged it in, hit the power nothing happened.\nI opened her up and had a look and say nothing much. A little orange\nLED comes on when I plug her in but that's it, after some googling\nI found some reference to re-seating all the parts, but no change.\nThe problem I'm having is that since the power supply is some Dell\nspecific one, ATX block with what looks like one of the old AT\npower connectors, I cant figure out weather this is a Mobo prob\nor a PSU prob. Just to futily try and drag this back OT, I want\nto install Linux on it when I get it working. If anyone knows\nwhat the problem might be give me a shout.Cheers,\nPeter.--\nPeter Aherne, Software Engineer, \nMotorola Ireland Ltd.\nPh: +353 21 4511234 Mobile: +353 87 2246834-- \nIrish Linux Users' Group: ilug@linux.ie\nhttp://www.linux.ie/mailman/listinfo/ilug for (un)subscription information.\nList maintainer: listmaster@linux.ie"
                                Output:
                                {{
                                    "explanation": "Technical support inquiry sent to a public mailing list. It contains a verifiable signature block with standard contact details, lacks any fabricated urgency or requests for sensitive information, and the included URL is a standard, safe mailing list administrative link. While the email has some spelling errors, it does not contain any of the common indicators of phishing such as suspicious links, requests for sensitive information, or unrealistic offers. The tone and content are consistent with a legitimate technical support request.",
                                    "risk_level": "Low",
                                    "is_phishing": false
                                }}

                                --- EXAMPLE 2: Phishing Email ---
                                Email: 'emailing : mon , 30 aug 2004 19 : 28 : 52 - 0800 dear sir / madam ; from our records we understand that you are inquiring about a new profession . we have a limited , ont time offer . our university can offer you a pre - qualified degree in your field of choice . we offer signing bonuses of up to $ 15 , 000 in your profession . to obtain your degree with valid transcripts & information on new career bonusus , follow our link : sincerely , dr . julie paige administration office this communication is of private matter . if you have received this and ar enot the individual or group it may concern or show interest in this please follow so we know http : / / 1 highereducation . com / st . html and the proper measures will proficiently expidited in a timely manner .'
                                Output:
                                {{
                                    "explanation": "Uses a generic greeting, contains multiple spelling and grammatical errors ('ont time', 'bonusus', 'expidited'), offers highly big unrealistic financial incentive ('$ 15 , 000') for a 'pre-qualified degree'. Too good to be true. Directs the victim to a suspicious, unofficial domain. The email also mentions 'limited on time offer' which is a common tactic to create urgency and pressure the recipient into acting without due consideration. All of these are strong indicators of a phishing attempt.",
                                    "risk_level": "High",
                                    "is_phishing": true
                                }}

                                --- EXAMPLE 3: Phishing Email ---
                                Email: "returned mail : see transcript for details the original message was received at tue , 19 jul 2005 07 : 06 : 09 - 0400 from root @ localhost - - - - - the following addresses had permanent fatal errors - - - - - antique ( reason : can \' t create ( user ) output file ) ( expanded from : ) - - - - - transcript of session follows - - - - - procmail : quota exceeded while writing " / var / spool / mail / antique " 550 5 . 0 . 0 antique . . . can \' t create output"
                                Output:
                                {{
                                    "explanation": "This is a classic 'Fake NDR' (Non-Delivery Report) or 'Quota Exceeded' spoof. The email mimics an automated server error ('fatal errors', 'quota exceeded') to create sudden panic or confusion. Phishers frequently spoof these administrative bounce messages to trick victims into clicking a malicious link to 'upgrade their quota' or 'view the undelivered message', bypassing the victim's natural skepticism of external emails.",
                                    "risk_level": "High",
                                    "is_phishing": true
                                }}

                                --- EXAMPLE 4: Safe Email ---
                                Email: "please grant access : application request ( kker - 4 r 3 klb ) security resource line item request kker - 4 r 3 klb has been submitted for your processing . to view the request , click your left mouse button on the notes document link below ."
                                Output:
                                {{
                                    "explanation": "Appears to be a legitimate, automated internal IT or ticketing system notification. It includes a highly specific, standard-formatted reference tracking code ('kker - 4 r 3 klb') and uses dry, administrative language. It does not threaten negative consequences or create artificial urgency. The instruction to click a 'notes document link' is standard operational practice for legacy enterprise environments (like Lotus Notes).",
                                    "risk_level": "Low",
                                    "is_phishing": false
                                }}
                                
                                --- EXAMPLE 5: Safe Email ---
                                Email: 'hpl nom for may 11 , 2001 ( see attached file : hplno 511 . xls ) - hplno 511 . xls'
                                Output:
                                {{
                                    "explanation": "This is a routine, automated or highly abbreviated B2B transactional email. It uses specific operational shorthand ('hpl nom', indicating a daily business nomination) paired with a specific date. The attached spreadsheet ('hplno 511 . xls') directly correlates with the date (May 11 -> 5/11) and the context of the message. It contains zero social engineering indicators, no artificial urgency, and no requests for credentials, pointing to a standard, safe internal workflow.",
                                    "risk_level": "Low",
                                    "is_phishing": false
                                }}

                                --- ACTUAL TASK ---
                                Analyse this email for phishing indicators.
                                Email: "{email['Email Text']}"

                                CRITICAL FINAL INSTRUCTION: You MUST return a valid JSON object. Do not output any conversational text. Use this exact structure:
                                {{
                                    "explanation": "Step-by-step reasoning of the technical and social indicators...",
                                    "risk_level": "Low" | "Medium" | "High",
                                    "is_phishing": boolean
                                }}
                            """,
                            "format": "json",
                            "stream": False,
                            "options": {
                                "temperature": 0.4,
                                "num_ctx": 4096
                            }
                        }

                        try:
                            response = requests.post(api_url, json=payload)
                            response.raise_for_status()
                            
                            raw_response = response.json().get("response", "").strip()
                            thinking_text = response.json().get("thinking", "").strip()

                            full_text = raw_response if raw_response else thinking_text

                            start_idx = full_text.find('{')
                            end_idx = full_text.rfind('}')
                            
                            if start_idx != -1 and end_idx != -1 and end_idx > start_idx:
                                json_string = full_text[start_idx:end_idx + 1]
                            else:
                                raise ValueError("No JSON brackets found in the AI response.")

                            verdict = json.loads(json_string)

                            # --- DEBUGGING PRINTS ---
                            raw_val = verdict.get('is_phishing')
                            print(f"\n[DEBUG] Perspective: {perspective['role_name']}")
                            print(f"[DEBUG] Full JSON Dict: {verdict}")
                            print(f"[DEBUG] Value of 'is_phishing': {raw_val}")
                            print(f"[DEBUG] Python Data Type: {type(raw_val)}\n")
                            # ------------------------

                            if "is_phishing" not in verdict:
                                raise ValueError(f"Model returned valid JSON, but forgot the keys! Raw dict: {verdict}")

                            # --- NEW: Tag the verdict with the role that generated it ---
                            verdict['perspective_used'] = perspective['role_name']

                            collected_verdicts.append(verdict)
                            success = True
                            break  

                        except requests.exceptions.RequestException as e:
                            print(f"API Error on email {i}, path {perspective['role_name']} (Attempt {attempt+1}/{max_retries}): {e}")
                        except json.JSONDecodeError as e:
                            print(f"JSON Error on email {i}, path {perspective['role_name']} (Attempt {attempt+1}/{max_retries}).")
                        except Exception as e:
                            print(f"Unexpected Error on email {i}, path {perspective['role_name']} (Attempt {attempt+1}/{max_retries}): {e}")
                
                    if not success:
                        print(f"-> Path {perspective['role_name']} for email {i} completely failed.")
                
                # --- SYNTHESIS STAGE: Tally the votes ---
                if len(collected_verdicts) == 0:
                    writer.writerow([email['Unnamed: 0'], email['Email Type'], "ERROR", "ERROR", "Failed all paths"])
                else:
                    # Calculate the majority
                    phishing_votes = sum(1 for v in collected_verdicts if v.get('is_phishing') is True)
                    safe_votes = len(collected_verdicts) - phishing_votes
                    
                    # Explicit Tie-Breaking Logic
                    if phishing_votes > safe_votes:
                        majority_is_phishing = True
                    elif safe_votes > phishing_votes:
                        majority_is_phishing = False
                    else:
                        # IT IS A TIE (e.g., 1 to 1). Enforce Zero Trust Policy.
                        majority_is_phishing = True
                    
                    # Find the first dictionary in our list that voted for the winning side
                    winning_response = next(v for v in collected_verdicts if v.get('is_phishing') == majority_is_phishing)
                    
                    # Log the vote count AND the specific persona that provided the winning logic!
                    vote_tally_note = f"[Vote: {phishing_votes} Phish / {safe_votes} Safe | Winning Logic: {winning_response.get('perspective_used')}] "
                    final_explanation = vote_tally_note + winning_response.get('explanation', 'Missing explanation')

                    writer.writerow([
                        email['Unnamed: 0'], 
                        email['Email Type'], 
                        majority_is_phishing, 
                        winning_response.get('risk_level', 'Missing'), 
                        final_explanation
                    ])
                    
    print(f"\nAnalysis complete.\nResults saved to '{save_folder}'\n")

In [30]:
model_list = ['gemma3:4b', 'llama3:latest', 'qwen3.5:4b', 'gemma3:12b']
# model_list = ['mixtral:8x7b']
email_list = sample2
save_folder = "Model Evaluation 8b - Self-Consistency with Multi-Perspective Prompting with 0.4 Temperature"

test_models(email_list, model_list, save_folder)
evaluate_models(model_list, save_folder)

===Models selected: ['gemma3:4b', 'llama3:latest', 'qwen3.5:4b', 'gemma3:12b']===

Starting Multi-Perspective analysis...
Email count: 2
Model: gemma3:4b



  0%|          | 0/2 [00:00<?, ?it/s]


[DEBUG] Perspective: Network/Technical Analyst
[DEBUG] Full JSON Dict: {'explanation': "The email exhibits several red flags. Firstly, the language is overly enthusiastic and uses marketing jargon ('breathing new life', 'powerful marketing tools', 'breath of fresh air', 'click away from your future success'). Secondly, the URL is obfuscated with underscores and lacks a clear domain. Thirdly, the email attempts to create a sense of urgency and opportunity by promising success. While it doesn't contain a direct link, the phrasing strongly suggests a link exists and is intended to entice the recipient. The use of 'checkour prices and hot offers' is also a common tactic in phishing emails. The email also contains numerous spelling errors, which is a common characteristic of phishing attempts.", 'risk_level': 'High', 'is_phishing': True}
[DEBUG] Value of 'is_phishing': True
[DEBUG] Python Data Type: <class 'bool'>


[DEBUG] Perspective: Social Engineering Expert
[DEBUG] Full JSON Dict: {'e

 50%|█████     | 1/2 [04:22<04:22, 262.20s/it]


[DEBUG] Perspective: Fraud/Compliance Investigator
[DEBUG] Full JSON Dict: {'explanation': "The email exhibits several red flags. It uses overly enthusiastic and hyperbolic language ('breathing new life', 'powerful marketing tools', 'stand out among the competitors', 'click away from your future success'). The offer of 'custom design' and 'hot offers' is vague and lacks specifics. The email contains numerous grammatical errors and typos, which is a common tactic used in phishing emails to mask their true intent. The call to action ('click here') is generic and doesn't provide a clear destination. While not overtly threatening, the email attempts to create a sense of urgency and opportunity, a common manipulation technique. The overall tone is overly promotional and lacks any verifiable details or legitimate business context. The use of 'ioqos' is also suspicious and unverified.", 'risk_level': 'High', 'is_phishing': True}
[DEBUG] Value of 'is_phishing': True
[DEBUG] Python Data Type: 

...


[DEBUG] Perspective: Network/Technical Analyst
[DEBUG] Full JSON Dict: {'explanation': "The email exhibits several concerning indicators. Firstly, the grammar and spelling are poor ('front - endlogo', 'carefui', 'wili', 'toois', 'brinq'). This is a common tactic used by phishers to bypass spam filters or to appear less sophisticated and more 'human'. The email uses emotionally manipulative language ('breathing new life', 'future success', 'stand out among the competitors') to entice the recipient. It offers a service (logo and visual identity design) without a clear, verifiable company affiliation. The URL, though not immediately visible, is likely to redirect to a malicious or phishing site. The 'not interested' link is a common phishing tactic to track opens and engagement, even if the recipient doesn't click the primary link. The overall tone and lack of professional polish strongly suggest a phishing attempt.", 'risk_level': 'High', 'is_phishing': True}
[DEBUG] Value of 'is_phishi

 50%|█████     | 1/2 [11:57<11:57, 717.80s/it]


[DEBUG] Perspective: Fraud/Compliance Investigator
[DEBUG] Full JSON Dict: {'explanation': 'The email exhibits several red flags indicative of a phishing attempt. Firstly, the language is overly promotional and uses hyperbolic phrases like "breathing new life into your business" and "future success," which are common in phishing scams designed to entice recipients. Secondly, the email contains numerous typos and grammatical errors (e.g., "ioqos", "carefui", "wili", "brinq"), which are often present in phishing emails due to lack of professional oversight. The offer of custom design services for logos, stationery, and websites is generic and lacks any personalization or specific details about the recipient\'s business. The call to action, "click here to see the samples of our artwork, check our prices and hot offers," is a standard tactic to lure victims to a potentially malicious website. The inclusion of a "not interested" opt-out link is a common technique used by phishers to make t

100%|██████████| 2/2 [22:00<00:00, 660.05s/it]


[DEBUG] Perspective: Fraud/Compliance Investigator
[DEBUG] Full JSON Dict: {'explanation': "This email appears to be a discussion thread from a mailing list or forum, likely related to linguistics or etymology. It contains a complex topic with specific terminology ('lucus a non lucendo', 'origines sive etymologiae'), referencing historical sources and individuals (St. Isidore of Seville, John T. Waterman). The tone is academic and conversational, characteristic of online forums. There are no requests for credentials, financial transactions, or urgent actions. The email lacks any indicators of social engineering or malicious intent. While the language is dense and technical, this is consistent with the subject matter and doesn't suggest a phishing attempt. The email's content and style are consistent with a legitimate online discussion.", 'risk_level': 'Low', 'is_phishing': False}
[DEBUG] Value of 'is_phishing': False
[DEBUG] Python Data Type: <class 'bool'>


Analysis complete.
Result

In [31]:
import os
import csv
import json
import requests
from tqdm import tqdm

def test_models(email_list, model_list, save_folder):
    api_url = "http://localhost:11434/api/generate"
    os.makedirs(save_folder, exist_ok=True)

    print(f"===Models selected: {model_list}===")

    for model_name in model_list:
        print(f'\nStarting Multi-Perspective analysis...\nEmail count: {email_list.num_rows}\nModel: {model_name}\n')
        
        csv_file = os.path.join(save_folder, f"{model_name.replace(':', '_')}.csv")
        with open(csv_file, mode='w', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow(["Email", "Label", "is_phishing_verdict", "risk_level", "explanation"])
            
            for i in tqdm(range(email_list.num_rows)):
                email = email_list[i]
                
                # --- MULTI-PERSPECTIVE SETTINGS ---
                # Define the 3 specific "hats" the model must wear
                perspectives = [
                    {
                        "role_name": "Network/Technical Analyst",
                        "instruction": "Focus heavily on technical indicators: scrutinize URLs, look for spoofed domains, weird routing data, or unusual file attachment types."
                    },
                    {
                        "role_name": "Social Engineering Expert",
                        "instruction": "Focus heavily on psychological manipulation: look for artificial urgency, authoritative pressure, generic greetings, or emotional triggers designed to bypass logic."
                    },
                    {
                        "role_name": "Fraud/Compliance Investigator",
                        "instruction": "Focus heavily on objective risk: look for requests for sensitive credentials, unusual financial transactions, or attempts to bypass standard corporate verification procedures."
                    }
                ]
                
                collected_verdicts = []
                
                # Run the email through the model 3 times, using a different perspective each time
                for perspective in perspectives:
                    max_retries = 5
                    success = False

                    for attempt in range(max_retries):
                        payload = {
                            "model": model_name,
                            "system": f"""
                                You are an expert cybersecurity analyst. Your only task is to analyze an email and determine if it is phishing or safe.
                                
                                CURRENT ROLE CONSTRAINT: You must analyze this email strictly from the perspective of a {perspective['role_name']}. 
                                {perspective['instruction']}
                                
                                CRITICAL RULES:
                                1. ZERO TRUST POLICY: If an email contains ANY suspicious elements (e.g., unexpected links, slight urgency, unusual requests), flag it as phishing. Do not give the sender the benefit of the doubt.
                                2. Flag subtle manipulation: Even if an email looks like standard corporate communication or an automated newsletter, flag it if it attempts to harvest credentials, bypass standard procedures, or manipulate the recipient.
                                3. Do not assume an email is safe just because it lacks an obvious malicious link; pure social engineering attempts must be flagged.
                                4. If ambiguous or you are unsure, default to Phishing ("is_phishing": true).
                                5. You must return a raw JSON object and absolutely nothing else.
                            """,
                            "prompt": f"""
                                Here are examples of how to classify emails:

                                --- EXAMPLE 1: Phishing Email ---
                                Email: "returned mail : see transcript for details the original message was received at tue , 19 jul 2005 07 : 06 : 09 - 0400 from root @ localhost - - - - - the following addresses had permanent fatal errors - - - - - antique ( reason : can \' t create ( user ) output file ) ( expanded from : ) - - - - - transcript of session follows - - - - - procmail : quota exceeded while writing " / var / spool / mail / antique " 550 5 . 0 . 0 antique . . . can \' t create output"
                                Output:
                                {{
                                    "explanation": "This is a classic 'Fake NDR' (Non-Delivery Report) or 'Quota Exceeded' spoof. The email mimics an automated server error ('fatal errors', 'quota exceeded') to create sudden panic or confusion. Phishers frequently spoof these administrative bounce messages to trick victims into clicking a malicious link to 'upgrade their quota' or 'view the undelivered message', bypassing the victim's natural skepticism of external emails.",
                                    "risk_level": "High",
                                    "is_phishing": true
                                }}

                                --- EXAMPLE 2: Safe Email ---
                                Email: "please grant access : application request ( kker - 4 r 3 klb ) security resource line item request kker - 4 r 3 klb has been submitted for your processing . to view the request , click your left mouse button on the notes document link below ."
                                Output:
                                {{
                                    "explanation": "Appears to be a legitimate, automated internal IT or ticketing system notification. It includes a highly specific, standard-formatted reference tracking code ('kker - 4 r 3 klb') and uses dry, administrative language. It does not threaten negative consequences or create artificial urgency. The instruction to click a 'notes document link' is standard operational practice for legacy enterprise environments (like Lotus Notes).",
                                    "risk_level": "Low",
                                    "is_phishing": false
                                }}
                                
                                --- EXAMPLE 3: Safe Email ---
                                Email: 'hpl nom for may 11 , 2001 ( see attached file : hplno 511 . xls ) - hplno 511 . xls'
                                Output:
                                {{
                                    "explanation": "This is a routine, automated or highly abbreviated B2B transactional email. It uses specific operational shorthand ('hpl nom', indicating a daily business nomination) paired with a specific date. The attached spreadsheet ('hplno 511 . xls') directly correlates with the date (May 11 -> 5/11) and the context of the message. It contains zero social engineering indicators, no artificial urgency, and no requests for credentials, pointing to a standard, safe internal workflow.",
                                    "risk_level": "Low",
                                    "is_phishing": false
                                }}

                                --- ACTUAL TASK ---
                                Analyse this email for phishing indicators.
                                Email: "{email['Email Text']}"

                                CRITICAL FINAL INSTRUCTION: You MUST return a valid JSON object. Do not output any conversational text. Use this exact structure:
                                {{
                                    "explanation": "Step-by-step reasoning of the technical and social indicators...",
                                    "risk_level": "Low" | "Medium" | "High",
                                    "is_phishing": boolean
                                }}
                            """,
                            "format": "json",
                            "stream": False,
                            "options": {
                                "temperature": 0.4,
                                "num_ctx": 4096
                            }
                        }

                        try:
                            response = requests.post(api_url, json=payload)
                            response.raise_for_status()
                            
                            raw_response = response.json().get("response", "").strip()
                            thinking_text = response.json().get("thinking", "").strip()

                            full_text = raw_response if raw_response else thinking_text

                            start_idx = full_text.find('{')
                            end_idx = full_text.rfind('}')
                            
                            if start_idx != -1 and end_idx != -1 and end_idx > start_idx:
                                json_string = full_text[start_idx:end_idx + 1]
                            else:
                                raise ValueError("No JSON brackets found in the AI response.")

                            verdict = json.loads(json_string)

                            # --- DEBUGGING PRINTS ---
                            raw_val = verdict.get('is_phishing')
                            print(f"\n[DEBUG] Perspective: {perspective['role_name']}")
                            print(f"[DEBUG] Full JSON Dict: {verdict}")
                            print(f"[DEBUG] Value of 'is_phishing': {raw_val}")
                            print(f"[DEBUG] Python Data Type: {type(raw_val)}\n")
                            # ------------------------

                            if "is_phishing" not in verdict:
                                raise ValueError(f"Model returned valid JSON, but forgot the keys! Raw dict: {verdict}")

                            # --- NEW: Tag the verdict with the role that generated it ---
                            verdict['perspective_used'] = perspective['role_name']

                            collected_verdicts.append(verdict)
                            success = True
                            break  

                        except requests.exceptions.RequestException as e:
                            print(f"API Error on email {i}, path {perspective['role_name']} (Attempt {attempt+1}/{max_retries}): {e}")
                        except json.JSONDecodeError as e:
                            print(f"JSON Error on email {i}, path {perspective['role_name']} (Attempt {attempt+1}/{max_retries}).")
                        except Exception as e:
                            print(f"Unexpected Error on email {i}, path {perspective['role_name']} (Attempt {attempt+1}/{max_retries}): {e}")
                
                    if not success:
                        print(f"-> Path {perspective['role_name']} for email {i} completely failed.")
                
                # --- SYNTHESIS STAGE: Tally the votes ---
                if len(collected_verdicts) == 0:
                    writer.writerow([email['Unnamed: 0'], email['Email Type'], "ERROR", "ERROR", "Failed all paths"])
                else:
                    # Calculate the majority
                    phishing_votes = sum(1 for v in collected_verdicts if v.get('is_phishing') is True)
                    safe_votes = len(collected_verdicts) - phishing_votes
                    
                    # Explicit Tie-Breaking Logic
                    if phishing_votes > safe_votes:
                        majority_is_phishing = True
                    elif safe_votes > phishing_votes:
                        majority_is_phishing = False
                    else:
                        # IT IS A TIE (e.g., 1 to 1). Enforce Zero Trust Policy.
                        majority_is_phishing = True
                    
                    # Find the first dictionary in our list that voted for the winning side
                    winning_response = next(v for v in collected_verdicts if v.get('is_phishing') == majority_is_phishing)
                    
                    # Log the vote count AND the specific persona that provided the winning logic!
                    vote_tally_note = f"[Vote: {phishing_votes} Phish / {safe_votes} Safe | Winning Logic: {winning_response.get('perspective_used')}] "
                    final_explanation = vote_tally_note + winning_response.get('explanation', 'Missing explanation')

                    writer.writerow([
                        email['Unnamed: 0'], 
                        email['Email Type'], 
                        majority_is_phishing, 
                        winning_response.get('risk_level', 'Missing'), 
                        final_explanation
                    ])
                    
    print(f"\nAnalysis complete.\nResults saved to '{save_folder}'\n")

In [33]:
model_list = ['gemma3:4b', 'llama3:latest', 'qwen3.5:4b', 'gemma3:12b']
# model_list = ['mixtral:8x7b']
email_list = sample2
save_folder = "Model Evaluation 8c - Self-Consistency with Multi-Perspective Prompting with 0.4 Temperature and 3 Examples"

test_models(email_list, model_list, save_folder)
evaluate_models(model_list, save_folder)

===Models selected: ['gemma3:4b', 'llama3:latest', 'qwen3.5:4b', 'gemma3:12b']===

Starting Multi-Perspective analysis...
Email count: 2
Model: gemma3:4b



  0%|          | 0/2 [00:00<?, ?it/s]


[DEBUG] Perspective: Network/Technical Analyst
[DEBUG] Full JSON Dict: {}
[DEBUG] Value of 'is_phishing': None
[DEBUG] Python Data Type: <class 'NoneType'>

Unexpected Error on email 0, path Network/Technical Analyst (Attempt 1/5): Model returned valid JSON, but forgot the keys! Raw dict: {}

[DEBUG] Perspective: Network/Technical Analyst
[DEBUG] Full JSON Dict: {'explanation': 'This email exhibits several red flags. The language is overly enthusiastic and uses phrases like "breathing new life," "powerful marketing tools," and "click away from your future success," which are common tactics in phishing attempts to create urgency and excitement. The URL is completely obfuscated with random characters ("ioqos"). The email lacks a clear sender domain and uses a generic greeting ("us thinking of"). The attachment name ("artwork") is suspicious and could contain malware. The email\'s overall tone and content are inconsistent with typical professional branding and design services. The use of

 50%|█████     | 1/2 [04:41<04:41, 281.97s/it]


[DEBUG] Perspective: Fraud/Compliance Investigator
[DEBUG] Full JSON Dict: {'explanation': 'This email exhibits several red flags. The language is overly enthusiastic and uses phrases like "breathing new life," "powerful marketing tools," and "click away from your future success," which are common tactics in phishing emails designed to create urgency and manipulate the recipient. The offer of "custom design of ioqos, stationery and web-sites" is vague and lacks specific details, a common characteristic of fraudulent services. The email also contains a large block of underscores, which is often used to obscure malicious links. While the email doesn\'t directly ask for credentials, the overall tone and content strongly suggest a deceptive attempt to lure the recipient into a scam. The lack of a clear sender and the use of marketing jargon further raise suspicion.', 'risk_level': 'High', 'is_phishing': True}
[DEBUG] Value of 'is_phishing': True
[DEBUG] Python Data Type: <class 'bool'>




...

 50%|█████     | 1/2 [11:19<11:19, 679.29s/it]


[DEBUG] Perspective: Fraud/Compliance Investigator
[DEBUG] Full JSON Dict: {'explanation': "This email exhibits several red flags indicative of a phishing attempt. Firstly, the language is overly promotional and uses hyperbolic phrasing like 'breathing new life into your business,' 'powerful marketing tools,' and 'future success.' This is a common tactic to entice recipients. Secondly, the email offers services (logo design, stationery, websites) unsolicited, which is unusual for legitimate business communications. The misspelling of 'careful' as 'carefui' and 'marketing' as 'marketinq' introduces a level of unprofessionalism that is atypical of established businesses. The inclusion of 'click here' with a link (not visible in the provided text, but implied) is a standard phishing technique to redirect users to a malicious website. The 'not interested' opt-out link is a common tactic to make the email appear legitimate while simultaneously tracking recipients who click it, further enab

100%|██████████| 2/2 [22:35<00:00, 677.98s/it]


[DEBUG] Perspective: Fraud/Compliance Investigator
[DEBUG] Full JSON Dict: {'explanation': "This email appears to be a chain of replies within a discussion group or forum. It contains a seemingly random topic about linguistic evolution and references other postings and individuals ('sue morrish', 'benji wald', 'john t waterman'). The language is informal and conversational, characteristic of online discussions. There are no requests for credentials, unusual financial transactions, or attempts to bypass corporate verification procedures. The subject line ('re: 6.293 words that are their own opposites') is unusual but consistent with a discussion topic. While the email is lengthy, its content is benign and does not exhibit any hallmarks of phishing or social engineering. The references to external sources (books, individuals) suggest a genuine, albeit quirky, discussion.", 'risk_level': 'Low', 'is_phishing': False}
[DEBUG] Value of 'is_phishing': False
[DEBUG] Python Data Type: <class '


Model: gemma3:12b. Results evaluated: 2.

	Accuracy: 1.0
	Precision: 1.0
	Recall: 1.0
	F1 Score: 1.0

===Model evaluation complete. Comparison saved to 'Model Evaluation 8c - Self-Consistency with Multi-Perspective Prompting with 0.4 Temperature and 3 Examples\model_comparison.csv'===


# Iteration 9: Simulated Tree-of-Thought Prompting

In [ ]:
import os
import csv
import json
import requests
from tqdm import tqdm

def test_models(email_list, model_list, save_folder):
    api_url = "http://localhost:11434/api/generate"
    os.makedirs(save_folder, exist_ok=True)

    print(f"===Models selected: {model_list}===")

    for model_name in model_list:
        print(f'\nStarting Simulated Tree-of-Thought analysis...\nEmail count: {email_list.num_rows}\nModel: {model_name}\n')
        
        csv_file = os.path.join(save_folder, f"{model_name.replace(':', '_')}.csv")
        with open(csv_file, mode='w', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            # CHANGE 1: Added "debate_transcript" to the CSV header so you can read the experts' arguments!
            writer.writerow(["Email", "Label", "is_phishing_verdict", "risk_level", "explanation", "debate_transcript"])
            
            for i in tqdm(range(email_list.num_rows)):
                email = email_list[i]

                max_retries = 5
                success = False

                for attempt in range(max_retries):
                    payload = {
                        "model": model_name,
                        "system": """
                            You are an expert cybersecurity system simulating a panel of analysts. Your task is to analyze an email and determine if it is phishing or safe.
                            
                            CRITICAL RULES:
                            1. ZERO TRUST POLICY: If an email contains ANY suspicious elements (e.g., unexpected links, slight urgency, unusual requests), flag it as phishing. Do not give the sender the benefit of the doubt.
                            2. Flag subtle manipulation: Even if an email looks like standard corporate communication, flag it if it attempts to harvest credentials or manipulate the recipient.
                            3. If ambiguous or you are unsure, default to Phishing ("is_phishing": true).
                            4. You must return a raw JSON object and absolutely nothing else.
                        """,
                        "prompt": f"""
                            Imagine three different cybersecurity experts (A Network Analyst, A Social Engineering Expert, and A Fraud Investigator) are answering this question.
                            All experts will write down 1 step of their thinking, then share it with the group.
                            Then all experts will go on to the next step, discussing and debating the evidence.
                            If any expert realizes they are wrong at any point, they leave the discussion.
                            Finally, the remaining experts will synthesize a final verdict.
                            
                            Here are examples of how to classify emails:

                            --- EXAMPLE 1: Safe Email ---
                            Email: "Ok, Iknow this is blatantly OT but I'm beginning to go insane.\nHad an old Dell Dimension XPS sitting in the corner and decided to\nput it to use..."
                            Output:
                            {{
                                "expert_debate_transcript": "Network Analyst: The headers and links point to a standard linux mailing list. Social Expert: The tone is informal but matches a public forum IT request. No manufactured urgency. Fraud Investigator: There are no requests for money or credentials. Network Analyst: I agree, this looks entirely benign.",
                                "explanation": "Technical support inquiry sent to a public mailing list lacking any fabricated urgency or requests for sensitive information.",
                                "risk_level": "Low",
                                "is_phishing": false
                            }}

                            --- EXAMPLE 2: Phishing Email ---
                            Email: 'emailing : mon , 30 aug 2004 19 : 28 : 52 - 0800 dear sir / madam ; from our records we understand that you are inquiring about a new profession . we have a limited , ont time offer ... signing bonuses of up to $ 15 , 000 ... follow our link http : / / 1 highereducation . com / st . html'
                            Output:
                            {{
                                "expert_debate_transcript": "Network Analyst: The URL provided uses a suspicious numerical subdomain '1 highereducation'. Social Expert: The greeting is completely generic ('dear sir / madam') and creates intense artificial urgency ('limited, ont time offer'). Fraud Investigator: Offering an unsolicited $15,000 signing bonus is a classic too-good-to-be-true financial hook. Social Expert: The spelling errors ('bonusus', 'expidited') confirm this is an untargeted scam.",
                                "explanation": "Uses a generic greeting, contains multiple spelling errors, offers highly unrealistic financial incentives, and directs the victim to a suspicious, unofficial domain.",
                                "risk_level": "High",
                                "is_phishing": true
                            }}

                            --- EXAMPLE 3: Phishing Email ---
                            Email: "returned mail : see transcript for details the original message was received at tue , 19 jul 2005 07 : 06 : 09 - 0400 from root @ localhost - - - - - the following addresses had permanent fatal errors - - - - - antique ( reason : can \' t create ( user ) output file ) ( expanded from : ) - - - - - transcript of session follows - - - - - procmail : quota exceeded while writing " / var / spool / mail / antique " 550 5 . 0 . 0 antique . . . can \' t create output"
                            Output:
                            {{
                                "expert_debate_transcript": "Network Analyst: The headers claim to be from root@localhost, simulating a system daemon, but the structure is a known fake NDR template. Social Expert: It leverages panic by using terms like 'fatal errors' and 'quota exceeded' to rush the user. Fraud Investigator: The end goal here is almost certainly a malicious link designed to steal credentials under the guise of 'fixing' the quota issue. Network Analyst: Agreed, this is a spoofed administrative alert.",
                                "explanation": "This is a classic 'Fake NDR' or 'Quota Exceeded' spoof designed to create sudden panic and trick users into clicking malicious links.",
                                "risk_level": "High",
                                "is_phishing": true
                            }}

                            --- EXAMPLE 4: Safe Email ---
                            Email: "please grant access : application request ( kker - 4 r 3 klb ) security resource line item request kker - 4 r 3 klb has been submitted for your processing . to view the request , click your left mouse button on the notes document link below ."
                            Output:
                            {{
                                "expert_debate_transcript": "Network Analyst: The specific tracking code format ('kker - 4 r 3 klb') is indicative of an automated ticketing system. Social Expert: The language is completely dry and administrative. It lacks any artificial urgency or threats of negative consequences. Fraud Investigator: There is no request for sensitive credentials or money, just an instruction to view a standard Notes document. Network Analyst: Everything points to a routine internal IT notification.",
                                "explanation": "Appears to be a legitimate, automated internal IT or ticketing system notification using standard administrative language and tracking codes.",
                                "risk_level": "Low",
                                "is_phishing": false
                            }}
                            
                            --- EXAMPLE 5: Safe Email ---
                            Email: 'hpl nom for may 11 , 2001 ( see attached file : hplno 511 . xls ) - hplno 511 . xls'
                            Output:
                            {{
                                "expert_debate_transcript": "Network Analyst: The attachment name ('hplno 511 . xls') directly correlates with the date in the text ('may 11'). Social Expert: The email is incredibly brief and contains zero manipulative language or urgency. Fraud Investigator: This is purely transactional. There is no financial or credential risk present in the text itself. Network Analyst: This is a safe, standard automated B2B operational email.",
                                "explanation": "A routine, highly abbreviated B2B transactional email with an attachment that matches the stated date and operational shorthand.",
                                "risk_level": "Low",
                                "is_phishing": false
                            }}

                            --- ACTUAL TASK ---
                            Conduct the multi-expert debate and classify this email:
                            Email: "{email['Email Text']}"

                            Return a raw JSON object with this exact structure:
                            {{
                                "expert_debate_transcript": "Script the step-by-step debate between the Network Analyst, Social Expert, and Fraud Investigator here...",
                                "explanation": "A concise summary of the winning experts' final conclusion.",
                                "risk_level": "Low" | "Medium" | "High",
                                "is_phishing": boolean
                            }}
                        """,
                        "format": "json", 
                        "stream": False,
                        "options": {
                            "temperature": 0.4, # CHANGE 2: Raised slightly so the "experts" don't all say the exact same thing
                            "num_ctx": 4096
                        }
                    }

                    try:
                        response = requests.post(api_url, json=payload)
                        response.raise_for_status()
                        
                        raw_response = response.json().get("response", "").strip()
                        thinking_text = response.json().get("thinking", "").strip()

                        full_text = raw_response if raw_response else thinking_text

                        start_idx = full_text.find('{')
                        end_idx = full_text.rfind('}')
                        
                        if start_idx != -1 and end_idx != -1 and end_idx > start_idx:
                            json_string = full_text[start_idx:end_idx + 1]
                        else:
                            raise ValueError("No JSON brackets found in the AI response.")

                        verdict = json.loads(json_string)

                        # CHANGE 3: Extracted the new debate transcript and added it to the row being written
                        writer.writerow([
                            email['Unnamed: 0'], 
                            email['Email Type'], 
                            verdict['is_phishing'], 
                            verdict.get('risk_level', 'Missing'), 
                            verdict.get('explanation', 'Missing'),
                            verdict.get('expert_debate_transcript', 'Missing')
                        ])

                        success = True
                        break 

                    except requests.exceptions.RequestException as e:
                        print(f"API Error on email {i} (Attempt {attempt+1}/{max_retries}): {e}")
                    except json.JSONDecodeError as e:
                        print(f"JSON Error on email {i} (Attempt {attempt+1}/{max_retries}). Hallucination: {json_string[:50]}...")
                    except Exception as e:
                        print(f"Unexpected Error on email {i} (Attempt {attempt+1}/{max_retries}): {e}")
            
                if not success:
                    print(f"-> Skipping email {i} after {max_retries} failed attempts.")
                    # CHANGE 4: Added an extra "ERROR" string to ensure the CSV columns stay perfectly aligned if it fails
                    writer.writerow([email['Unnamed: 0'], email['Email Type'], "ERROR", "ERROR", "Failed after max retries", "ERROR"])
    
        print(f"\nAnalysis complete.\nResults saved to '{csv_file}'\n")

In [ ]:
model_list = ['gemma3:4b', 'llama3:latest', 'qwen3.5:4b', 'gemma3:12b']
model_list = ['mixtral:8x7b']
email_list = sample2
save_folder = "Model Evaluation 9 - Simulated Tree-of-Thought Prompting"

test_models(email_list, model_list, save_folder)
evaluate_models(model_list, save_folder)